In [ ]:
from pathlib import Path
import sys,os,json,subprocess,hashlib,zipfile
from google.colab import files
base=Path('/content/ns64_clean')
blocked=['/content/drive/MyDrive','/content/ns64','/content/imc_ns64_colab_migration.zip','/content/imc_ets_images.zip','/content/private.json']
assert not any(Path(p).exists() for p in blocked),'Use a freshly deleted and reconnected runtime without Drive'
subprocess.run(['nvidia-smi','--query-gpu=name,driver_version','--format=csv,noheader'],check=True)
bundle=Path('/content/imc_ns64_clean_runtime.zip')
if not bundle.exists(): files.upload()
assert hashlib.sha256(bundle.read_bytes()).hexdigest()=='39e9bb97ede83c777a06f385fbe4acb27fc4665ba1556b504dc64000dcafb925','Wrong runtime ZIP'
base.mkdir(exist_ok=True)
with zipfile.ZipFile(bundle) as z:z.extractall(base)
bootstrap=base/'bootstrap'
subprocess.run([sys.executable,'-m','pip','install','--target',str(bootstrap),'uv==0.13.8'],check=True)
setup_env=os.environ.copy()
setup_env.update(PYTHONPATH=str(bootstrap),UV_PYTHON_INSTALL_DIR=str(base/'python'),UV_CACHE_DIR=str(base/'uv_cache'))
uv=[sys.executable,'-m','uv']
subprocess.run(uv+['python','install','3.11.11'],env=setup_env,check=True)
python=base/'venv/bin/python'
if not python.exists():subprocess.run(uv+['venv','--python','3.11.11','--seed',str(base/'venv')],env=setup_env,check=True)
subprocess.run([str(python),'-m','pip','install','setuptools==75.8.0','wheel==0.45.1'],check=True)

subprocess.run([str(python),'-m','pip','install','--no-deps','--extra-index-url','https://download.pytorch.org/whl/cu121','-r',str(base/'requirements.txt')],check=True)


In [ ]:
import urllib.request,shutil
manifest={'assets':json.loads((base/'assets.json').read_text())}
asset_dir=base/'weights'
asset_dir.mkdir(exist_ok=True)
def download(url,path,expected):
    path.parent.mkdir(parents=True,exist_ok=True)
    if path.exists():
        with path.open('rb') as handle: actual=hashlib.file_digest(handle,'sha256').hexdigest()
        assert actual==expected,path
        return
    partial=path.with_suffix(path.suffix+'.partial')
    print('Downloading',path.name,flush=True)
    digest=hashlib.sha256()
    with urllib.request.urlopen(url,timeout=120) as response,partial.open('wb') as output:
        while chunk:=response.read(4*1024**2):
            digest.update(chunk);output.write(chunk)
    assert digest.hexdigest()==expected,path
    partial.replace(path)
for key,a in manifest['assets'].items():
    if key=='dinov2':
        path=asset_dir/'dinov2'/a['filename']
        url='https://huggingface.co/facebook/dinov2-base/resolve/main/pytorch_model.bin'
        for name,h in a['sidecars'].items():
            download('https://huggingface.co/facebook/dinov2-base/resolve/main/'+name,path.parent/name,h)
    else:path=asset_dir/a['filename'];url=a['url']
    download(url,path,a['sha256'])
print('All seven asset hashes verified.')


In [ ]:
result=subprocess.run([str(python),'-I','-c',"import json, sys, os\nfrom pathlib import Path\nimport torch, pycolmap, asmk.hamming, curope, kornia, mast3r.model, dust3r.inference, yaml\nbase=Path('/content/ns64_clean')\nassert torch.__version__=='2.5.1+cu121' and torch.cuda.is_available() and pycolmap.__version__=='3.11.1'\nassets=json.loads((base/'assets.json').read_text())\nsource=base/'source'\naliases=yaml.safe_load((source/'conf/models.yaml').read_text())\nfor key,asset in assets.items():\n    path=base/'weights'/('dinov2' if key=='dinov2' else asset['filename'])\n    assert path.exists()\n    aliases[asset['alias']]={'kernel':str(path),'local':str(path)}\nalias_file=base/'aliases.json'\nalias_file.write_text(json.dumps(aliases))\nos.environ.update(DEFAULT_MODEL_LIST_PATH=str(alias_file),DEFAULT_EXTRA_DIR=str(base/'extra'),MPLBACKEND='Agg',HF_HUB_OFFLINE='1')\nsys.path.insert(0,str(source))\nfrom pipeline import create_pipeline\nfrom config import PipelineConfig\nconfig=PipelineConfig.load_config(source/'conf/pipeline/imc2025/mast3rhybrid/mast3rhybrid-022-c.yaml')\nprint(json.dumps(dict(status='DEPENDENCIES_PASS',python=sys.version,torch=torch.__version__,device=torch.cuda.get_device_name(0),config=config.type)))\n"],cwd=base,capture_output=True,text=True)
print(result.stdout)
print(result.stderr[-4000:])
result.check_returncode()


In [ ]:
from google.colab import files
assert not Path('/content/ns64_neutral_runs').exists(), 'Use a fresh runtime'
if not Path('/content/imc2024_dioscuri_baalshamin_opaque.zip').is_file():
    files.upload()
assert Path('/content/imc2024_dioscuri_baalshamin_opaque.zip').is_file()

In [ ]:
WORKER='import os, sys, hashlib, json, time, resource, zipfile, functools, contextlib\nfrom unittest.mock import patch\nfrom pathlib import Path\nwork=Path.cwd()\nseed=int(sys.argv[1])\nbase=Path(\'/content/ns64_clean\')\nsource=base/\'source\'\ndef save(name, value):\n    temp=work/(name+\'.tmp\')\n    temp.write_text(json.dumps(value,indent=2))\n    temp.replace(work/name)\ndef require(condition,message):\n    if not condition:\n        raise RuntimeError(message)\ndef digest(path):\n    with path.open(\'rb\') as stream:\n        return hashlib.file_digest(stream,\'sha256\').hexdigest()\nPINS={20261111:\'1b7bc97f6f54a2f5daa967a06e67211cd565f84d2d9403cd60f1faa50af543b0\'}\nASSET_MANIFEST=\'696df0470cebc09a532d45466819b43bd363656402b761a23bd6a9ee403b0ca0\'\nexec(\'SOURCE_HASHES = {\\\'clusterings/__init__.py\\\': \\\'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855\\\', \\\'clusterings/base.py\\\': \\\'7beef8d00e75692e3930e28e768aa89d0c0357578ba5d2a3b139966e93b66ee7\\\', \\\'clusterings/config.py\\\': \\\'0e3c1acf07244827e2e26812c6e4dbe299699bfd6a0ce1e8fb86ff8c751009c5\\\', \\\'clusterings/connected_component.py\\\': \\\'c1f93a8e7765bff8b7745c54640d17d036d9e4f1be5ef3b157796e36a2862836\\\', \\\'clusterings/dbscan.py\\\': \\\'d675e9cf30590bdaa08b8c5216ece765adb2b253ff7a74c36144d7ec80fbdf0a\\\', \\\'clusterings/debug.py\\\': \\\'3d8862e439981f8a278dd71c1ecf050312558ff8d73adf2c964236775d5be9bb\\\', \\\'clusterings/factory.py\\\': \\\'0bfdc9323b9561b67a932359cfef21f665a4142980c50457ca21b36cda434689\\\', \\\'clusterings/fps.py\\\': \\\'d358e2b823fa1f1ad7d60ec964e5ed62aabe1c3a47e8748269e367677855e6d8\\\', \\\'clusterings/fps_mast3r.py\\\': \\\'18039387fa9e1cdf6d0a51d203c668b3597e96d5532458349d6068e115efbe9b\\\', \\\'clusterings/vggt.py\\\': \\\'b9012686793bdac51e2e4196da210664e4ce43e15da7056c5668994ef51560f8\\\', \\\'clusterings/visualizer.py\\\': \\\'54dd16b1a6baa3bf50e5ed58434c9e1a5bd6133141ff285380e12fb48c841a64\\\', \\\'colmap.py\\\': \\\'295249258e089f5809d480d35fdfb4b0106d5e6031fb3571aa3782a5b6e0f19b\\\', \\\'conf/models.yaml\\\': \\\'02bc8c06f1bfe99f649a19a1350a7d991d3d97d08d411a89abc6055ed3496e44\\\', \\\'conf/pipeline/imc2025/mast3rhybrid/mast3rhybrid-022-c.yaml\\\': \\\'e7829be60865192d3ddcc8555b94bc8678415e265a566d8ce8386ef17b0cc763\\\', \\\'config.py\\\': \\\'f626086c3881c75f84ae64b5ddfd6be74baf354dd6d8d581589f2aa955111580\\\', \\\'core.py\\\': \\\'2750b1c6b8c5a8f725bc449744a5c6229c53b60c7a9a3ee4d2eb78fdc3e66d41\\\', \\\'data.py\\\': \\\'4429d63ca587b71c1ef46faa608818cbd61457816500e0a3b6a8207b82022734\\\', \\\'data_schema.py\\\': \\\'3cf2d510a9bf331fbbb29ee308814b07a078cecc3a78966ceffd28ec572d4487\\\', \\\'distributed.py\\\': \\\'7c4be6628e9dd5e92df9b53306d85a5334c98aac04efd7c12b73d51abebf8362\\\', \\\'extractor.py\\\': \\\'f34dfd33a284dfddd8c839c9b612c3e1c9bfbbf1a7b1fd2a093cef9fee0cfa6f\\\', \\\'features/_research_only/__init__.py\\\': \\\'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855\\\', \\\'features/_research_only/magicleap_superpoint.py\\\': \\\'f32399db8f22427735bde8f6011728d8d9e8fe84d2f45e86e02f021b1d035813\\\', \\\'features/_research_only/silk.py\\\': \\\'ebe2a3785852ee10e25aae8581924146b70018103f2a58778f4905c1c02154d8\\\', \\\'features/alike.py\\\': \\\'dcd419f6749d935a02ed83983c804e017e3d1c07ac23610abaf90c7c431a2c99\\\', \\\'features/aliked.py\\\': \\\'827da2137b1e4f203a58938d5d0158e9d55ef8fedb8eb8b1c119cabae3eaf3c5\\\', \\\'features/base.py\\\': \\\'7c0be80130988d292170c6f3592b00b799584115cb83d11806167b78af141dea\\\', \\\'features/config.py\\\': \\\'30af327f342918c4198a4d6a432270996551af43d1301704ccb7f99f64102f25\\\', \\\'features/dad.py\\\': \\\'08f367951d3bab19deac860ba502319e0e30ebddcd7e5e3c4333eabf86278575\\\', \\\'features/darkfeat.py\\\': \\\'6b87d9508b04f2329100530d240098b43250d9dd83e65ce6a62a74567cbfa85f\\\', \\\'features/dedode.py\\\': \\\'b1afca381a95b35859bece1a493d0dbcecf4336a7c4e6acd7c2f27f4744bc254\\\', \\\'features/delf_pytorch.py\\\': \\\'bc440a2c1582370d01be25982e0cbbcc1d19aeaaede47803bd521bfb926d1673\\\', \\\'features/disk.py\\\': \\\'b3b118791b3e7e9186da7f8331441c22474b4461b2c1912df4b8f4a66b374ebe\\\', \\\'features/edgepoint2.py\\\': \\\'00708a7b4d090e4fcba255cf080b58d4f6a1f1456fe5ffed1ab4e680dc883d25\\\', \\\'features/factory.py\\\': \\\'e007b11649b304219ddd841fc80d33aa40223f89e42a6ea49b1baa6ebcb371c0\\\', \\\'features/featurebooster.py\\\': \\\'24ee10db13bf374d332626094f70b38b0169564e62d91644bbc42c2e68bff1fb\\\', \\\'features/gim/superpoint.py\\\': \\\'d1b48cc84df7b91e607c007ca379f66989fc47d849996b224485583f3d50b688\\\', \\\'features/hardnet.py\\\': \\\'96bf60cedec21453b6481469e0fea0325dc53ede6a86eb8d989af95ed2c43d35\\\', \\\'features/hardnet8.py\\\': \\\'74a7e8a1debd408fbdff28bfd7a380138e3c36d18e7ec04c2fb4aaee958edc2c\\\', \\\'features/lanet.py\\\': \\\'f1fde61fd806903e78d9a88e76ba24a39851fcf22cdd20eabcd8a1caa5cb0f16\\\', \\\'features/lightglue/aliked.py\\\': \\\'98d701c2790bdf6a2002c661a68c5ee5878770602f5530261dc810cf547dd221\\\', \\\'features/lightglue/aliked_dedode_v2.py\\\': \\\'a49b4706bbe5803c1a0077ffe255b807131218cc19b021f74049681f690575e0\\\', \\\'features/lightglue/aliked_tile.py\\\': \\\'b8f6b14fa3946150a46587ca0952f5cfdbcab098c94246a6ac4de976b908e163\\\', \\\'features/lightglue/disk.py\\\': \\\'7b81d1ccc9fb61653e52c1f0a004e044a2454943bb5392646ae7a8572ed6766d\\\', \\\'features/lightglue/doghardnet.py\\\': \\\'587a7351b5db5e150a1e75f83e4be7b49b6a229f0d1effdfc2f2f25c695d4c1a\\\', \\\'features/lightglue/sift.py\\\': \\\'3463692c362ba8c4f4a7b42826774d5644ce969d0700c8a399e64173690285e8\\\', \\\'features/lightglue/superpoint.py\\\': \\\'b0d2a2723d48e0ce553886a226e985db34b4892faa1a3261628820bc5df9efc9\\\', \\\'features/line2d.py\\\': \\\'eb2d903f8e9c74eb0bc634fd4071cbacdb03e9a5748afff9e40b287eef5cab93\\\', \\\'features/mtldesc.py\\\': \\\'67d49646491e01ceb76d3d00d9260ff2832d0225d90c3503e183815c579d9ada\\\', \\\'features/multiscale.py\\\': \\\'98e393956818dd9730795db4e6fd62418125618a138630af42b1f8e8d93ab99a\\\', \\\'features/posfeat.py\\\': \\\'4dd3e5cbf4561ad71d9453efbbd64d1220733b572bd63e0c501f2a063ac88571\\\', \\\'features/relf.py\\\': \\\'e0a8e4758cb17cba92502842e1eb036e346fbf45daa47419d7f6d8a8cfbcde0b\\\', \\\'features/sfd2.py\\\': \\\'b1c7d3b1db36915fee9fbd8d0561281f24a3dbfd0d868d5995de5fdd8154b25e\\\', \\\'features/superpoint.py\\\': \\\'a7ad33696296ae24c25eadceac8072d57c8ca55f1df945b38a876b84b146e1e2\\\', \\\'features/visualizer.py\\\': \\\'59c29b1e87fd27c5b9edce7b25777eac6e42dbf92f52322c38e114e28e4dd5d8\\\', \\\'features/xfeat.py\\\': \\\'494aaf9a525fb1de9aaa7b31c168df77443c05d87d9c50c3ec4d1aaee5f7a4ad\\\', \\\'global_descriptors/__init__.py\\\': \\\'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855\\\', \\\'global_descriptors/apgem.py\\\': \\\'d7a1907218b6a2b8288bec6fdb2ff2362f5b5d741faaac84ff492f3c5fcb12bc\\\', \\\'global_descriptors/base.py\\\': \\\'153dee5e63e386aa33f7c3b75f2e50f1a6bb03996dba3c8ce48e79fed1df5cef\\\', \\\'global_descriptors/config.py\\\': \\\'3b8db177a507e541705d31a8798b17871540fd6a43527e83d5a78ff92b5d0b17\\\', \\\'global_descriptors/cvnet.py\\\': \\\'99de3f5a8e64ce36e9e03c1dd4f8cd963fef4750f6d7039f8cde3590ca3d278a\\\', \\\'global_descriptors/dinov2.py\\\': \\\'1f9dd33c56ea469f15d3ae4ebaf54622e93e58f37c18e90dd797e6ee148a5025\\\', \\\'global_descriptors/dinov2_salad.py\\\': \\\'e8a68a383b235ff7fa222b662a8dc6234df9105d87fcda675d86b69a0a624e09\\\', \\\'global_descriptors/isc.py\\\': \\\'82a9a78920945bca1ee3adad418837ef1243667c3eac7b68c0c6209cd34c70b4\\\', \\\'global_descriptors/mast3r_retrieval_asmk.py\\\': \\\'99959b58718b611ed64e55d0ff457b8bf7446f4e99fcc249f0734c62170e92b7\\\', \\\'global_descriptors/mast3r_retrieval_spoc.py\\\': \\\'98c8168402a294ac0567a392f05d9f786f03364c88d7e511b98e4044f2025b60\\\', \\\'global_descriptors/mast3r_spoc.py\\\': \\\'90d8d3b804019a3a7da04285e250eb0bb73e5ab8e001962b7288f59c330939f7\\\', \\\'global_descriptors/moge_depth_hog.py\\\': \\\'b94c6da9dec71a35f5fd80f80c334b898c0d84c3fe647f23ac69692f85b4163d\\\', \\\'global_descriptors/moge_dinov2.py\\\': \\\'a16a54a088f2922972af6fc3e3d64a2c79c20c96d76e8df01c82c08a1be68403\\\', \\\'global_descriptors/moge_feature.py\\\': \\\'fa452eb4f4d49b0f9e4055d467afbfc5d7eb19b44deeee50f06629df02b4db93\\\', \\\'global_descriptors/patch_netvlad.py\\\': \\\'6885211a3af045af3ffa15b4c1e66a288290b537ee8a4a7adf28458daf093fdf\\\', \\\'global_descriptors/siglip2.py\\\': \\\'a5578c4abdcf686945c2ec40cfb74aa346f4c3a5b2384d16655cc5e250f81861\\\', \\\'kernel.py\\\': \\\'f31a7640a3d3616e2067e86cc361cf46b922ebab1c89c8a1176cc31ad7945d8d\\\', \\\'localizers/base.py\\\': \\\'d3abffc81731b0f3a775bb8d3828cc115dc3c12e90c40bd371c882ab69f8710c\\\', \\\'localizers/common.py\\\': \\\'2befe73e13eeabe85b6b73ddd4480a30d738caa15dc908d1a953910f616cc793\\\', \\\'localizers/config.py\\\': \\\'fc7d26a4a9f0da18bef1aeda685e70b5d0b6f6571873aa1367db79cef945c600\\\', \\\'localizers/factory.py\\\': \\\'7e0d2a4daa2cf4a0d88254fa2a4fd7bdb57c3043dae7f654798bcf1d35862033\\\', \\\'localizers/mapfree.py\\\': \\\'c7f779dd8fc709c5f375998eb2c96d1a6d0b1a41b0c5b096385bf8615e3512c6\\\', \\\'localizers/mickey.py\\\': \\\'9c1a46f3ff6fb28984a7258761413e3c97b2dd9c41d360fae78f05a7f78a8656\\\', \\\'localizers/two_view.py\\\': \\\'25a711b77bfa2991cfef6c8dfc83ff263343d44849bc97522728caaea6f2291c\\\', \\\'matchers/__init__.py\\\': \\\'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855\\\', \\\'matchers/_research_only/__init__.py\\\': \\\'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855\\\', \\\'matchers/_research_only/aspanformer.py\\\': \\\'dae916b49d42070a913309010794a66d2830eee29d234232c02902ca384a81bf\\\', \\\'matchers/_research_only/magicleap_superglue.py\\\': \\\'23be97927e6d5093c7459f583314ef4e981d15f7aa14313c3459c5813d3fd28a\\\', \\\'matchers/_research_only/magicleap_superglue_rot.py\\\': \\\'902c0acaee7c0091d130b9ac603d0d0f0977c6b5ffe35d0fabb8a70ce4abc056\\\', \\\'matchers/adamatcher.py\\\': \\\'13f6dd5a0659db99311c6c2ecf787b84d308005b0c4f5e41ea062652af0ac366\\\', \\\'matchers/base.py\\\': \\\'36a16d43e290faf358ef0367993031eb8abbbc3103b43b1a7a209466e8c55557\\\', \\\'matchers/config.py\\\': \\\'7c1d516b116f4b56eac5789be3000034d087dd98521c893beeb1ffdc3d1a4f9f\\\', \\\'matchers/dkm.py\\\': \\\'ec74f011c38c91ef1e29ffdb17c024fec0fcab4631d5013b00bdc13349304928\\\', \\\'matchers/dkm_rot.py\\\': \\\'1a39aac9e0b20f09f2f8f1d7cb513e01d71843fd72238311dcad2c52cc461ba6\\\', \\\'matchers/dual_softmax.py\\\': \\\'8316b364f25643c937679831749196b83aec0748dc9bddd13b4dda2765921b2f\\\', \\\'matchers/ecotr.py\\\': \\\'66791ca7a312ccfcd9d924bfe2566a456fc11f3ea971709c373396ba108e9680\\\', \\\'matchers/efficientloftr.py\\\': \\\'db1bb27651702a8a7d1d9ff2c29a41ebd53076b1682855f11df1b08ce927730f\\\', \\\'matchers/factory.py\\\': \\\'64758b9fabc54b7eb09ba805eae211f4bdae572cd137200e100c1e3118aeaecc\\\', \\\'matchers/gim_dkm.py\\\': \\\'ef9071cd9ec26ae82010e11a6e1068656291992540ec0e13ad0beab39476655b\\\', \\\'matchers/gim_lightglue.py\\\': \\\'8c0675eb8cf643b17e1536aa316af2c343362717449527ed6a8e97a3863b6857\\\', \\\'matchers/gim_loftr.py\\\': \\\'55260b351751d9e0efed2950e2a44b3382244b3059190df87e96bf40f724e13a\\\', \\\'matchers/lightglue.py\\\': \\\'4d02072cc53bb504df9cbf980567f8594dd49015d658269e034894371633c7b6\\\', \\\'matchers/line2d.py\\\': \\\'2c295e66fadfc6adc0a4063b6e0acbaebab3b32907633f952026a1f34bd4d2a6\\\', \\\'matchers/loftr.py\\\': \\\'c889d5d7350a28dc02e058b081762fb793429b38603fcccb2580179cef629385\\\', \\\'matchers/mast3r.py\\\': \\\'edbfaa69ff76439b6ce0d35052491495c50edc9032af5b18ab149ac00233a50c\\\', \\\'matchers/mast3r_c2f.py\\\': \\\'a501284fa1966bb49056a66a5591f7da354bd7747e217fa07c98cd8cc6ec2bee\\\', \\\'matchers/mast3r_hybrid.py\\\': \\\'966d8f2dce49298d8e36d2639dd5ab3f6116d19516d54b66678822d3a6c448b4\\\', \\\'matchers/mast3r_mpsfm.py\\\': \\\'03ae29e0c994dfe299cf88514ee40dbd0c241eda9211eee04345d95a751fb6e6\\\', \\\'matchers/mast3r_mpsfm_sparse.py\\\': \\\'d11767f177220c6e821f0513b3d26a44c069edd1dabae4adaf5f88a4f5cbee64\\\', \\\'matchers/mast3r_sparse.py\\\': \\\'7014fb625268f144ed90e4a10cacdbc6b5a441acf72c5474eb36bb138a5e9ed8\\\', \\\'matchers/matchformer.py\\\': \\\'d40afb316cc806de769da50749b55addf5537e4a997472cbc1a0d048ee11ea8e\\\', \\\'matchers/nn_double_softmax.py\\\': \\\'bdeac1cc9fb8d69d13ceb6d2c69032d53b0de44ef63922487bce7d80f620128b\\\', \\\'matchers/nn_matcher.py\\\': \\\'0863ad2b577f2708fb75a29445803b49876e71151947ef1a5d9ae3e388f53974\\\', \\\'matchers/omniglue_onnx.py\\\': \\\'3198d0b8d502bd99371e059c9686e912c8f0d3360ead4bb930f091ea8b7362da\\\', \\\'matchers/openglue.py\\\': \\\'d2b2c08dc986156ee2bb1b1aad4b82babbc870ec682c6eedcc46c47695bcf9ff\\\', \\\'matchers/quadtree.py\\\': \\\'d57aabc04baf88b5ba8266fcb1f3e3359fd15206ba7065b74c6abbe25a22c40a\\\', \\\'matchers/roma.py\\\': \\\'8f7dadfeb7ea933de374d0a9072501488050ef0e994331da103ff68d59233e46\\\', \\\'matchers/se2loftr.py\\\': \\\'c47969274da828a296fdb28b2e0b73aabd5cd90a7299f7043521c41c911c1162\\\', \\\'matchers/steerer.py\\\': \\\'6f98c5d4d952ab775f768a9d4ea38bcc19d529a0aa120564692a4d89980a83f1\\\', \\\'matchers/vggt.py\\\': \\\'29456a4448e1468bcc98e1fcbd49b849af7ec1a7f7eacf99cb538c19733070fa\\\', \\\'matchers/visualizer.py\\\': \\\'719abdffe1ec801e0e6abc62242a29cf9fdfa5d06ee850b40c6560c27ecc4c9b\\\', \\\'matchers/xfeat.py\\\': \\\'a03eb413fa9d216147f253c0e02875d4cd7dacef3b0093016d86bffeae99d138\\\', \\\'models/__init__.py\\\': \\\'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855\\\', \\\'models/_research_only/__init__.py\\\': \\\'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855\\\', \\\'models/_research_only/magicleap/__init__.py\\\': \\\'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855\\\', \\\'models/_research_only/magicleap/matching.py\\\': \\\'b5e8301a714f66b100df4a86110844594853e5b0f53cd92192985a403341c8a9\\\', \\\'models/_research_only/magicleap/superglue.py\\\': \\\'20687b1dd58add93d423832e7067e82f8de53885c8ac2aa5e0b1d2f8272d2042\\\', \\\'models/_research_only/magicleap/superpoint.py\\\': \\\'66f9be4d5daf0452be37b0f2971f2004bc9417cb37015431bf241d78a51c7c2b\\\', \\\'models/_research_only/magicleap/utils.py\\\': \\\'690dde73ebc970d451de1cdf8305fdb8320a90346371fdd722349cfab8397c13\\\', \\\'models/_research_only/silk/__init__.py\\\': \\\'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855\\\', \\\'models/_research_only/silk/backbones/__init__.py\\\': \\\'9f2c336d52111ef51bae5b4959d8941968e9e26638c681de5388c58d3080624d\\\', \\\'models/_research_only/silk/backbones/abstract/__init__.py\\\': \\\'9f2c336d52111ef51bae5b4959d8941968e9e26638c681de5388c58d3080624d\\\', \\\'models/_research_only/silk/backbones/abstract/shared_backbone_multiple_heads.py\\\': \\\'1e88a1ffc698272de4f5e81021e81de4f91bf1c004fa46ec6a76af9dbf8d5885\\\', \\\'models/_research_only/silk/backbones/abstract/shared_backbone_multiple_heads_test.py\\\': \\\'1cf6b81872e1d7d24ad86dfc29ac91a576cc969675b4b6529049a80bc1b45907\\\', \\\'models/_research_only/silk/backbones/loftr/__init__.py\\\': \\\'9f2c336d52111ef51bae5b4959d8941968e9e26638c681de5388c58d3080624d\\\', \\\'models/_research_only/silk/backbones/loftr/linear_attention.py\\\': \\\'495913faec4d0bf0812f3ab5a80b897394440b010e238363a1ef80de1f3e3e91\\\', \\\'models/_research_only/silk/backbones/loftr/positional_encoding.py\\\': \\\'14a3c4e198aec956dcd577051ef32bad7ca18e3630077069f45e629607135055\\\', \\\'models/_research_only/silk/backbones/loftr/resnet_fpn.py\\\': \\\'2e9ac03e3d4e9bcf5eeaad9f15ff53203fe42d6735a51ea0be39121a30c1b46b\\\', \\\'models/_research_only/silk/backbones/loftr/transformer.py\\\': \\\'46759211184d08fd64c740bcf075548010d1f24be82b05ddb1b631874822b2da\\\', \\\'models/_research_only/silk/backbones/silk/__init__.py\\\': \\\'9f2c336d52111ef51bae5b4959d8941968e9e26638c681de5388c58d3080624d\\\', \\\'models/_research_only/silk/backbones/silk/coords.py\\\': \\\'0e88aef7b2cc500178e56e76dd5c355e981815ca9913abb1f1ea2088305b78d5\\\', \\\'models/_research_only/silk/backbones/silk/coords_test.py\\\': \\\'b1a1b29eaf24fa6a2bee59409df038207c6960e5994adf1f442c6177b849f9b1\\\', \\\'models/_research_only/silk/backbones/silk/silk.py\\\': \\\'4dcae22e855c3f900e21251bcfa48a63d9410f9c8aeaf1e423a2c3cfafb259d7\\\', \\\'models/_research_only/silk/backbones/superpoint/__init__.py\\\': \\\'9f2c336d52111ef51bae5b4959d8941968e9e26638c681de5388c58d3080624d\\\', \\\'models/_research_only/silk/backbones/superpoint/magicpoint.py\\\': \\\'db52caee7f4f1785e8d6d003ed6bd238e7ee91758016ad21d71cc3425eb7109d\\\', \\\'models/_research_only/silk/backbones/superpoint/superpoint.py\\\': \\\'762cc6a08641b654fc93604f5ddb580a728308afc8218170e843cf7a7b8027ce\\\', \\\'models/_research_only/silk/backbones/superpoint/utils.py\\\': \\\'45c2c56888c515b626f91d40d0375fcbb054026af5abebc43cbcf3b612bfe99e\\\', \\\'models/_research_only/silk/backbones/superpoint/utils_test.py\\\': \\\'988258c40f29ed92b6b7dce1addac407bd99b75d8480535b597d67f20c7360b7\\\', \\\'models/_research_only/silk/backbones/superpoint/vgg.py\\\': \\\'90f5dbda2e48208e6803f2877b9c097368ad27cb2e38d4ef27de45943d677999\\\', \\\'models/_research_only/silk/backbones/superpoint/vgg_test.py\\\': \\\'25d0b7c62af757971a7aab32e91de7212d94a2830c021e29584ac581d8ad29df\\\', \\\'models/_research_only/silk/backbones/unet/__init__.py\\\': \\\'9f2c336d52111ef51bae5b4959d8941968e9e26638c681de5388c58d3080624d\\\', \\\'models/_research_only/silk/backbones/unet/unet.py\\\': \\\'3c86f7986f89d28fe033a916aa335481c4047cbae4c3aa2852577ebbd0c13d68\\\', \\\'models/_research_only/silk/common.py\\\': \\\'fafa32cc1f089440014b0742ef0f0b08f851ffcf69fef24af198ce574f7c33d7\\\', \\\'models/_research_only/silk/config/__init__.py\\\': \\\'0ad029374663194002d36732c434b7a0a4bebe8c613a78242126a3a64f296f67\\\', \\\'models/_research_only/silk/config/constants.py\\\': \\\'111782a6a18708308142be26ac2a58c98f79941999d779690a02b96d6db3b3ae\\\', \\\'models/_research_only/silk/config/core.py\\\': \\\'15642b939d6dee102f14185472bd9506021e92bf528095cd79c0a6997b91d1ea\\\', \\\'models/_research_only/silk/config/formatter.py\\\': \\\'0c3039bb1c8913082f85111dacc7b896d7addc6b7ea2d2972c8704197b092a94\\\', \\\'models/_research_only/silk/config/formatter_test.py\\\': \\\'d31e75dbfdccf48de919fb821b3c2f7653f037d32b1ce272de6c7d51380f149d\\\', \\\'models/_research_only/silk/config/hardware.py\\\': \\\'04e02b89581f52add26beeb40b2d4934787bf8e2767ffffce76b6900f3e55c97\\\', \\\'models/_research_only/silk/config/model.py\\\': \\\'e7780fb7b5375c836e55905c94584c82ead1261150248d459196f552d2df67ff\\\', \\\'models/_research_only/silk/config/optimizer.py\\\': \\\'c5ed64419ff6289d58e122a71ab86fe3369a6dfe6f84cd6de51dc8a29596d13b\\\', \\\'models/_research_only/silk/config/optimizer_test.py\\\': \\\'8dbfbd982f850fe66f6980b59f7b123466d740508c10eafbf4b8b9f19113d322\\\', \\\'models/_research_only/silk/config/paths.py\\\': \\\'f47d55a80009f88a56316d14ef5598b2e82cc0232bcef297f0f9cef2d70c2bbf\\\', \\\'models/_research_only/silk/config/resolver.py\\\': \\\'6be6c6d57c01497191b7fb6a10825ae0f71fa0d3db39906d49d22746cf42a9d9\\\', \\\'models/_research_only/silk/config/sampler.py\\\': \\\'00093f9ccef1edc8d33dfe6ca34ac6797e6c9520dbf213a905fb4ff0ae1c1154\\\', \\\'models/_research_only/silk/cv/__init__.py\\\': \\\'9f2c336d52111ef51bae5b4959d8941968e9e26638c681de5388c58d3080624d\\\', \\\'models/_research_only/silk/cv/homography.py\\\': \\\'9e78b03bc8c8f71131cdfcf1247e5ba3f996630b1fb01902b7d4d0d6f6ecca87\\\', \\\'models/_research_only/silk/cv/homography_test.py\\\': \\\'d1dd86e3c4209e16f7e7530bca16f8fe898072c043f79719f45fd495f7d0ab38\\\', \\\'models/_research_only/silk/flow.py\\\': \\\'0fd4b08946cd77184b6bbad99f43142fd33a812e3dac31d7bd880e0c8ba4ed81\\\', \\\'models/_research_only/silk/losses/__init__.py\\\': \\\'9f2c336d52111ef51bae5b4959d8941968e9e26638c681de5388c58d3080624d\\\', \\\'models/_research_only/silk/losses/info_nce/__init__.py\\\': \\\'7596df3a6facc944c02ee7f15e2304c5fbf3685adad59de877120e08614ccecf\\\', \\\'models/_research_only/silk/losses/info_nce/loss.py\\\': \\\'ff14d6fec29d07c22bbd3e29ca795d39d230f36fdde4a8663ed4e9fcc9b4176e\\\', \\\'models/_research_only/silk/losses/info_nce/loss_test.py\\\': \\\'3b7dbb6c30bfadf6aab659a4f41d8c2818724f110de8789c42636683587cabeb\\\', \\\'models/_research_only/silk/losses/superpoint.py\\\': \\\'118e6a372b4616b8668f7c2d17b34da94c8e445a68831a0c26ef667afaf23363\\\', \\\'models/_research_only/silk/matching/mnn.py\\\': \\\'cfc4e279790a13a0da7601f0e4ea5d4a95bff3aaa258e9258ed02df30c01ae47\\\', \\\'models/_research_only/silk/models/__init__.py\\\': \\\'9f2c336d52111ef51bae5b4959d8941968e9e26638c681de5388c58d3080624d\\\', \\\'models/_research_only/silk/models/abstract.py\\\': \\\'aaeee82e6a9f6d17f445071a07e8849271609341ed8b79ad62bf9c5770ac9de1\\\', \\\'models/_research_only/silk/models/abstract_test.py\\\': \\\'33622181d9f4deecfd615217c077bd6f3e0064415e6abccd0797ac2b5bf21eb1\\\', \\\'models/_research_only/silk/models/magicpoint.py\\\': \\\'dc2bfb999cd8d81152956b32c1d858f91810ba708303bf4f5657833dd0ece560\\\', \\\'models/_research_only/silk/models/magicpoint_test.py\\\': \\\'9c693a4101bbf85dc24638b640ba90768ce7bb628f603d45a8a7f8e7324d5197\\\', \\\'models/_research_only/silk/models/pointtracker.py\\\': \\\'cc89385ef6653ee62fc297edc7bec18a52c52560688861984bdc9c14d8c3d3d9\\\', \\\'models/_research_only/silk/models/pointtracker_test.py\\\': \\\'f9b2f6f607e560bf25c58dedde70d635910d3adf87af0dee7ec6a684faedee67\\\', \\\'models/_research_only/silk/models/sift.py\\\': \\\'74ed30ca352a548d9be2ee1bd7917c9df5e858cd0b6dc79a18d8e8cd585c7efd\\\', \\\'models/_research_only/silk/models/silk.py\\\': \\\'674600006a3fafec8d789d59a60075f6a78f7d57d3f5f0bdc2964dde5ed9909d\\\', \\\'models/_research_only/silk/models/superpoint.py\\\': \\\'bf7306f3acbe09942e8d158a8373526f4cbdba03c2afb7c1302dde820e1f5b3f\\\', \\\'models/_research_only/silk/models/superpoint_test.py\\\': \\\'63347d77ab0f5295c7ea1e04276684f1e551476156f7de2436f3712616a32f64\\\', \\\'models/_research_only/silk/models/superpoint_utils.py\\\': \\\'1c80247e5cfbbfb6a10aeaca3a17b7992b6908f23557877b41f80efdc3742da1\\\', \\\'models/_research_only/silk/transforms/__init__.py\\\': \\\'9f2c336d52111ef51bae5b4959d8941968e9e26638c681de5388c58d3080624d\\\', \\\'models/_research_only/silk/transforms/abstract.py\\\': \\\'bad3e27e7b049d5d7d4608bd5ab5fb88a80ad2dcd4039a76065c6d246ff68320\\\', \\\'models/_research_only/silk/transforms/abstract_test.py\\\': \\\'9ca048832131af05fd2fa1ad7ce0b1e069136cc614ccbb9c145b9f9fddc422fc\\\', \\\'models/_research_only/silk/transforms/cv/__init__.py\\\': \\\'9f2c336d52111ef51bae5b4959d8941968e9e26638c681de5388c58d3080624d\\\', \\\'models/_research_only/silk/transforms/cv/homography.py\\\': \\\'5485779338c2497ee02fd656ad3b0e439810895113f00b3a9f246a759d1ad607\\\', \\\'models/_research_only/silk/transforms/cv/image.py\\\': \\\'59eb39a6df7bdff67084497ef571f836104eaeeb25655a49c4fc9fe22dac1dfd\\\', \\\'models/_research_only/silk/transforms/cv/image_test.py\\\': \\\'ce71a510804cd44ae1eba3087590bca273c12059fcb48d301f01a43677f1ccc2\\\', \\\'models/_research_only/silk/transforms/cv/video.py\\\': \\\'ba372a9da8c08d1ee1380c51de2b48e6dc819e05bde1c66f1628b075b9b04f3b\\\', \\\'models/_research_only/silk/transforms/dataset.py\\\': \\\'04af7f248b91e9e16f432ef5b9dda2d94c37ae8a6c1eb97e54b13d51904a7396\\\', \\\'models/_research_only/silk/transforms/metric.py\\\': \\\'7c25f3ca3d7a6d18a7c02d99a723b8e9b3fc7526dd345979bcc7f3418284dd31\\\', \\\'models/_research_only/silk/transforms/tensor.py\\\': \\\'1dd000a5e42b3b0034c8ecfccf594535213952f8bdb819fbdf166d8f0b4e4d58\\\', \\\'models/_research_only/silk/transforms/tensor_test.py\\\': \\\'dabf904ea81df415b91cce170e7f687ebc664a2ea5433ccb363815c3a0e1019e\\\', \\\'models/adamatcher/__init__.py\\\': \\\'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855\\\', \\\'models/adamatcher/adamatcher/__init__.py\\\': \\\'0371157f2d8e58eba6f2812284cd57e6ce6cfae7405ee53fbe85cac41b4dc1e1\\\', \\\'models/adamatcher/adamatcher/ada_module/__init__.py\\\': \\\'14e6ef32f604b66211a13331be391705237cc9d6f6bd29a2c82cef6f5f5d8d93\\\', \\\'models/adamatcher/adamatcher/ada_module/linear_attention.py\\\': \\\'22655ec769c3cd24cf149c1baafdaf821eabf51a097f764da893395af8eb9a37\\\', \\\'models/adamatcher/adamatcher/ada_module/transformer.py\\\': \\\'22756dc5b6cbe702ff2558192d6567191a83ee5b5a0edd4297c21816eb6355bc\\\', \\\'models/adamatcher/adamatcher/adamatcher.py\\\': \\\'0bbbcf1c9b2ad096d3a06fcc5953a31f704d21749903db05e731b8ec4afbcfaf\\\', \\\'models/adamatcher/adamatcher/backbone/__init__.py\\\': \\\'9cd954168a8797bfbd63249189dfcbd6fe2a948bd7c0b9e8885a6044111dd534\\\', \\\'models/adamatcher/adamatcher/backbone/feature_interaction.py\\\': \\\'c103e456745f32a00d54b3269559d7f781a51fbd0cee4f9dd1d7a97618281810\\\', \\\'models/adamatcher/adamatcher/backbone/resnet_fpn.py\\\': \\\'d8cecb5b874f9818dfe1b327299b7a3c7c1a4ddd0e7c40cb902f30e5fa5768d4\\\', \\\'models/adamatcher/adamatcher/utils/__init__.py\\\': \\\'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855\\\', \\\'models/adamatcher/adamatcher/utils/coarse_module.py\\\': \\\'d7cc9e3fe8d0970214a0f41c642606c088ed77d4102efe9e1f060e698806a44b\\\', \\\'models/adamatcher/adamatcher/utils/comm.py\\\': \\\'8cb4e046fa9488f5725c1fc856a7147420639109e1511cad1dc4985f878fd1a0\\\', \\\'models/adamatcher/adamatcher/utils/cvpr_ds_config.py\\\': \\\'a925cc2b60f6131a9d14168e24237ae6250259427e39e75de974ea8f690e1103\\\', \\\'models/adamatcher/adamatcher/utils/fine_module.py\\\': \\\'221d1ac90cbc6a4d6691dd4b20d1a410c890e6b8d0d4914e9a6e1461c3089a07\\\', \\\'models/adamatcher/adamatcher/utils/geometry.py\\\': \\\'193b142b476299033e3c371a146f8d91a42274a5c1bedb6b6af62588a8464595\\\', \\\'models/adamatcher/adamatcher/utils/optimization.py\\\': \\\'f38c0b77fbd8c277134d4529d3e267c500cdc03a2f4763cb1350fd3c0916cfd7\\\', \\\'models/adamatcher/adamatcher/utils/position_encoding.py\\\': \\\'9160675e5022dabf0c086995093b35a91a389e2048ec14cf8ed6e80d817441f2\\\', \\\'models/adamatcher/adamatcher/utils/supervision.py\\\': \\\'8288416554ba5fa53ded5bb34545510a1214be8814993ca827c2a85a2631e6e2\\\', \\\'models/adamatcher/adamatcher/utils/utils.py\\\': \\\'7ccc3327ca3a5242e574b6b0c6723eeca8a94399de1e0d4dcc08ee53743148f0\\\', \\\'models/adamatcher/config/default.py\\\': \\\'579d25e11280e664321b8ed5b827ea65a8ed37cf074d95e862ec643b8b35c4eb\\\', \\\'models/adamatcher/config/loftr_ds_dense.py\\\': \\\'a723a67efb32d0a8bc89ee030844598e6b82c1f1e8d8500245cdb6e294db41a0\\\', \\\'models/adamatcher/lightning/data.py\\\': \\\'df4520330d70df14ec096d783b247dfd3629915a0677f4eac9caf0a9be9fe0a7\\\', \\\'models/adamatcher/lightning/lightning_adamatcher.py\\\': \\\'d64afe068fc3d0ee46e9fd99cefcf26edc7df9db568eb03e02cb4731f5b5f9db\\\', \\\'models/adamatcher/utils/augment.py\\\': \\\'a58b649b39964733c9d225488bc9b448fffbe5e32873a6748330df2fe27c6cc5\\\', \\\'models/adamatcher/utils/comm.py\\\': \\\'1b974bce9e10795db6285d164477ae72c4a6d42225980e795f859657260e04be\\\', \\\'models/adamatcher/utils/dataloader.py\\\': \\\'395ee136007d7480079fc76edff9598e1202be757fd29fc88ba7314d6954d8c3\\\', \\\'models/adamatcher/utils/dataset.py\\\': \\\'116cbb56ed99718bc4672e6cd224b455a36345f14b4a05db4cbf781c8d8d8ecd\\\', \\\'models/adamatcher/utils/metrics.py\\\': \\\'dfb01289790a5d630e493647e7507f2337001a0114077422ef66877548ee58bd\\\', \\\'models/adamatcher/utils/misc.py\\\': \\\'0c0431a5e3146b9c41ff4d25f204e478262004d5b5b2f07b58801dfc1c1fee74\\\', \\\'models/adamatcher/utils/plotting.py\\\': \\\'15ab3e04b47618979c06aed64299ec4978a81dc8d419e52a415b0d28218e657e\\\', \\\'models/adamatcher/utils/profiler.py\\\': \\\'78b3e54d47d2d53884e4aa6d2dcb929ff5a915f15c8cb50c78226965cbae3415\\\', \\\'models/alike/__init__.py\\\': \\\'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855\\\', \\\'models/alike/alike.py\\\': \\\'5278c9661bc234c2b48b9e3cc42514a1f8157badcad2f1807db797750637cf28\\\', \\\'models/alike/alnet.py\\\': \\\'c49a809fda5536d27eeb656655d96ed48c4dc68333911c0908de8097022abcc0\\\', \\\'models/alike/soft_detect.py\\\': \\\'5ea5d0a8cd9d2a15f33c640a4c7b865716ab403fc1ff13dfecd33a57065ba8ba\\\', \\\'models/aliked/__init__.py\\\': \\\'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855\\\', \\\'models/aliked/nets/__init__.py\\\': \\\'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855\\\', \\\'models/aliked/nets/aliked.py\\\': \\\'086b3d7f4eb01910bc1e357749c571cd968405263536b7b1d6fe8ef1eadc8f97\\\', \\\'models/aliked/nets/blocks.py\\\': \\\'42e12589a437718147c55333b5adc9d9b2f90b4d039da6c6167c9498fdbe4779\\\', \\\'models/aliked/nets/padder.py\\\': \\\'3c2cdac5339a9c1c2b539cf624b8c1d7f1b321a60af76bf4e8e12a277edc5b6a\\\', \\\'models/aliked/nets/soft_detect.py\\\': \\\'b02633d9820f2035f5a4e9c7e288f920f2637be7878ce1e20293cdfe03c29044\\\', \\\'models/aspanformer/__init__.py\\\': \\\'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855\\\', \\\'models/aspanformer/aspanformer/__init__.py\\\': \\\'05bc02f403cb7a2743f3c0f0358a7f53edb1d62cd0019580a7dc571ef6199a43\\\', \\\'models/aspanformer/aspanformer/aspan_module/__init__.py\\\': \\\'d9a104ae4c5223620cb7f50659a43464a5e01edf45d5a1e0fc04825d5eae832d\\\', \\\'models/aspanformer/aspanformer/aspan_module/attention.py\\\': \\\'57280f9c36bae9f13d261d3a18541be2f74dde6f2a5272e0a4ee7ce9e560753d\\\', \\\'models/aspanformer/aspanformer/aspan_module/fine_preprocess.py\\\': \\\'1eeb83778a4c50b5752ff37d2b8ac72d76706ea4fe3bc222ab8c47263279fafe\\\', \\\'models/aspanformer/aspanformer/aspan_module/loftr.py\\\': \\\'a756f58fe2940eb775c87ead917ca68e6c81962297726f7ec21f634bdeb121d8\\\', \\\'models/aspanformer/aspanformer/aspan_module/transformer.py\\\': \\\'81effb2563f74dd8107f1b9dab9d49c67ba8d6a3ba77675e9c48568179661a1e\\\', \\\'models/aspanformer/aspanformer/aspanformer.py\\\': \\\'2b48a53be14ed227c36753e6dd9e690b0bd1e2e22cfa29db981a3ad41dcf3a8e\\\', \\\'models/aspanformer/aspanformer/backbone/__init__.py\\\': \\\'913859da2c9ba4b685c11d144cc9faefe1652b495ccaf081a41ffd466620021d\\\', \\\'models/aspanformer/aspanformer/backbone/resnet_fpn.py\\\': \\\'997d9778915524defd9f7e36b31992c811b85b215a13291af5ec781dc1a86061\\\', \\\'models/aspanformer/aspanformer/utils/coarse_matching.py\\\': \\\'7ff8d33f5b5b9637f6606c7e501d393301d5db273797f897ac4a525fb10657d3\\\', \\\'models/aspanformer/aspanformer/utils/cvpr_ds_config.py\\\': \\\'7f1153842d03a4a81f99b221952a13921905efada949abaa9a2148fac3e88b88\\\', \\\'models/aspanformer/aspanformer/utils/fine_matching.py\\\': \\\'823dd17053a8559f7b1ca32e41b46552c587c295ae1ab988ad75d2c91448cf3a\\\', \\\'models/aspanformer/aspanformer/utils/geometry.py\\\': \\\'d10afe85e7144777343ff193d16baab514683a19f207832422f15bc771226668\\\', \\\'models/aspanformer/aspanformer/utils/position_encoding.py\\\': \\\'541781f6ed0c5b158fcaa0bf3eb60fd996fb6de19fae74c592be851947c891e1\\\', \\\'models/aspanformer/aspanformer/utils/supervision.py\\\': \\\'8df9ce54c8cc5e35d2f4ce66574dba07fc2dce3a47b1ad2715b61c88b9666612\\\', \\\'models/aspanformer/config/default.py\\\': \\\'bdb4e1b098bf0eecfc378fca6035e848af4266be1bce5d96cb85507bc72f9eb2\\\', \\\'models/aspanformer/utils/augment.py\\\': \\\'0faf76ee3f598c8e1973178dc203a4342b82db838ee3634795f63324726e6ac3\\\', \\\'models/aspanformer/utils/comm.py\\\': \\\'b5a6eb8e91d95c946337b5657010abc76044a7ed0e8647f5171b8c5d0418b2ea\\\', \\\'models/aspanformer/utils/dataloader.py\\\': \\\'beeed315d335b37302d20cfe6beb9115cebdd796691d701a7c062d204be42730\\\', \\\'models/aspanformer/utils/dataset.py\\\': \\\'18c1361faae8c8d19ecf87afe8408e8cf8fbe0bf68e2723dd2ab6f6e5d897d83\\\', \\\'models/aspanformer/utils/metrics.py\\\': \\\'99c1fa489742608844a22098bb2c712f8ee26cea98916997721325628d346ef4\\\', \\\'models/aspanformer/utils/misc.py\\\': \\\'c1d8fe6db699a5ff35088784c6838d618ad00edc0f5d5e2733c0c17f5d81cedf\\\', \\\'models/aspanformer/utils/plotting.py\\\': \\\'4cbcb935cef7bc7bd49c0699d4a999bdf3e1c927ef2b3ed287bb80228cd41dfb\\\', \\\'models/aspanformer/utils/profiler.py\\\': \\\'e2d183e0be8c6edb168270be3eb1dfe6e5f56b4e8ca569b3ce16f574c78dbcf8\\\', \\\'models/config.py\\\': \\\'18217db6d4c985847d8c1ea631608d66aca89e910fdf1b65b97ccbc44d0e0689\\\', \\\'models/cvnet/__init__.py\\\': \\\'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855\\\', \\\'models/cvnet/core/CVNet_tester.py\\\': \\\'c1571af3bb6a7780696c899e6fa42006df349c2c229758bbd695f427079f259a\\\', \\\'models/cvnet/core/__init__.py\\\': \\\'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855\\\', \\\'models/cvnet/core/checkpoint.py\\\': \\\'532a3193054f773df7df1bdf094ef4324b7e28b68af0c6b3cebe89dc95e03c6e\\\', \\\'models/cvnet/core/config.py\\\': \\\'e0ce3924cac7cf48d6b1e0d4a9ee1dd33b6ddf6548e797510db3f49be6f7a4e9\\\', \\\'models/cvnet/core/transforms.py\\\': \\\'57e24aaa1ab7a41d8a51696031975bfaa366510b38adc2b2d2e028055dd72da6\\\', \\\'models/cvnet/model/CVNet_Rerank_model.py\\\': \\\'60769f79ec1bc95fcfd0328744430c8f88a18bf45109028fc5d50f29d1d649f2\\\', \\\'models/cvnet/model/CVlearner.py\\\': \\\'6974aef224407661ba14a4f233ae23c85bb947cd5b247bee70fb9127aaeec79b\\\', \\\'models/cvnet/model/__init__.py\\\': \\\'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855\\\', \\\'models/cvnet/model/base/__init__.py\\\': \\\'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855\\\', \\\'models/cvnet/model/base/conv4d.py\\\': \\\'4faedeed62b41a097ee1f34da62dd8f685faba551cc95f1b2b94b4e053df5ef5\\\', \\\'models/cvnet/model/base/correlation.py\\\': \\\'26e8d00d7b0d2c6a800d1c1d5a0ce1d844c3b0145bb74d612a0317f4b860dd73\\\', \\\'models/cvnet/model/base/feature.py\\\': \\\'7831021ce4066162be2a1797b983cd8ca2a37635c984c0c6e10a1b957c9dbe15\\\', \\\'models/cvnet/model/base/geometry.py\\\': \\\'86c3c0d0b127aff5f292b004088b6236f03cf7ba6fc90d42156bb466b21a3226\\\', \\\'models/cvnet/model/resnet.py\\\': \\\'3f4cefb8b52cd972c1ad1cca3b9007b6231c6d08834ad74cff846a3d1d87e1df\\\', \\\'models/dac/__init__.py\\\': \\\'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855\\\', \\\'models/dac/structure_tensor.py\\\': \\\'5de374377d23f5c76b278f2614716ac952d7584dc45bc1762b2fe74598346315\\\', \\\'models/darkfeat/misc.py\\\': \\\'6d885e3151f989a889803743d7c7bb4c45f6448b5883b6a936b398a5ae1d6ec1\\\', \\\'models/darkfeat/model.py\\\': \\\'61fa981f0412bfd68e8959ebcc26ae87936bbdbd7f0b4d22c68fdf11a06c4e66\\\', \\\'models/delf_pytorch/delf.py\\\': \\\'571dd9ba43f66ba70d68b62cb7886c474d3f09449401940c5ec1ad3ea3470605\\\', \\\'models/delf_pytorch/delf_helper.py\\\': \\\'01e98a8a226cd165f7748aa89f69786cc28aa8007961fd983909994cb2edd1a8\\\', \\\'models/delf_pytorch/extractor.py\\\': \\\'0c763cf5a343e13f8fe071cd8d8cbe21a43f5fcbefb1e033a5460619b4e6c64e\\\', \\\'models/delf_pytorch/layers.py\\\': \\\'cc153acd02eab18c510554fa5a73b2de33295ea5005ee0617280213b794490fa\\\', \\\'models/delf_pytorch/pca.py\\\': \\\'8360fbbd28cd9eee56c6f5edc48d7b401bea7e902a6edd0b1c91af52348d6b0a\\\', \\\'models/dinov2_salad/model.py\\\': \\\'9fddda5a992ef8cb01674c30b606cf3f70f94bc0d9937dce394ba9268dafc361\\\', \\\'models/dinov2_salad/models/__init__.py\\\': \\\'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855\\\', \\\'models/dinov2_salad/models/aggregators/__init__.py\\\': \\\'31a8f26a6659f49f8026c2e9592a144cb98c013458f29e449d8b5878c0e13a74\\\', \\\'models/dinov2_salad/models/aggregators/convap.py\\\': \\\'a047277aa625e7d7b0614fe7012c67592c7322514903b89d150c55acb9b66bd5\\\', \\\'models/dinov2_salad/models/aggregators/cosplace.py\\\': \\\'4ac957b2ccf6c0e7c462355fb5753b00311f9489c46061eb44da5dca15e4e33c\\\', \\\'models/dinov2_salad/models/aggregators/gem.py\\\': \\\'256098cdbadc9ade60f5edab96c3b4be61afbdd7ee58bd9361ed11ca638ab241\\\', \\\'models/dinov2_salad/models/aggregators/mixvpr.py\\\': \\\'91550d886c1a258f9782d76309c8e4ba8f99e27b9c853c1268148b32709484d2\\\', \\\'models/dinov2_salad/models/aggregators/salad.py\\\': \\\'00ef6181893f00169f0a21cb5757078790e3d1dd921ff4109c26482b870c55d4\\\', \\\'models/dinov2_salad/models/backbones/__init__.py\\\': \\\'086ee44cffbd98c66abfae65bccf582aab75d23173776bf793ed5f19d68c51fc\\\', \\\'models/dinov2_salad/models/backbones/dinov2.py\\\': \\\'0aae8146368f1674326cf66ae2aef591648f58c8dd01a9acce8fbaf3874d376a\\\', \\\'models/dinov2_salad/models/backbones/resnet.py\\\': \\\'4f5a0cde606edfa81a0454a9b18974cdc07135338a6bc4340ba09de5435a32c2\\\', \\\'models/dinov2_salad/models/dinov2/hub/__init__.py\\\': \\\'4cb84429008d894db3edcce826656c5bab6474bf57fb7852d15cb1768492f1f0\\\', \\\'models/dinov2_salad/models/dinov2/hub/backbones.py\\\': \\\'05bed1b479424492bfb0b16a4a2757c7f1330c462bdcb0f2a43c8d2110e16708\\\', \\\'models/dinov2_salad/models/dinov2/hub/classifiers.py\\\': \\\'9d3d847b58fe9966dd0ddf5607c0d338ef71d55ff8e1a4144dd15aec1b0cc804\\\', \\\'models/dinov2_salad/models/dinov2/hub/depth/__init__.py\\\': \\\'564026ad1d667c4c714a584f5e0bc9ba9da4aaf0e8eae79280f76c03c5997909\\\', \\\'models/dinov2_salad/models/dinov2/hub/depth/decode_heads.py\\\': \\\'19ec858f6d09cbbe774ef5b9e470edb4bc02d9818b414a2ad4127978ece7af0f\\\', \\\'models/dinov2_salad/models/dinov2/hub/depth/encoder_decoder.py\\\': \\\'24efb237ed2bc2338edb68d585997cbf4f6cfc8997468fb32b159120b3d66671\\\', \\\'models/dinov2_salad/models/dinov2/hub/depth/ops.py\\\': \\\'3b7126117bf38fe427084a92a4e2aff4c88415735d34307b83a27a208eae0fd6\\\', \\\'models/dinov2_salad/models/dinov2/hub/depthers.py\\\': \\\'8b3b8e7266b1585782b5b53e38122bf5d837f4d46347523d1296460824705b73\\\', \\\'models/dinov2_salad/models/dinov2/hub/utils.py\\\': \\\'9410fd10f64a003c638a0f6d2d64e65947d5a051bfc943d7e40edc3f160fdf0b\\\', \\\'models/dinov2_salad/models/dinov2/hubconf.py\\\': \\\'526247d8557808229847b3d4e881712a134f018d9710484a652bad18beb041b2\\\', \\\'models/dinov2_salad/models/dinov2/layers/__init__.py\\\': \\\'918196a6ce5c087cad373ec42652e17d5890f80a863943a064c779fe3e1c7445\\\', \\\'models/dinov2_salad/models/dinov2/layers/attention.py\\\': \\\'d87d6c1063e6e4021ef9a263abfdcabb9f5b4ae355d91b30ecd0d28bd55b0330\\\', \\\'models/dinov2_salad/models/dinov2/layers/block.py\\\': \\\'f2e118652d0cce4ead02e8c88f06a07db0754685395696df85aa6ae07c222b66\\\', \\\'models/dinov2_salad/models/dinov2/layers/dino_head.py\\\': \\\'21a3c10983280abab3103274e690531a6b97f5145d9d909f5b68863c9b972e84\\\', \\\'models/dinov2_salad/models/dinov2/layers/drop_path.py\\\': \\\'0e992756228930464d447ea1ab9337e463c8b2e6b265aef0639252d592bf2fef\\\', \\\'models/dinov2_salad/models/dinov2/layers/layer_scale.py\\\': \\\'46edeae6b1fce18c1a25a91efa9642604a200d0ba8d05b0e15c72cb8a53936b8\\\', \\\'models/dinov2_salad/models/dinov2/layers/mlp.py\\\': \\\'4253b84f60dbd82696d1c4868517ae6f992c3ddec37b0168061679bec203cd59\\\', \\\'models/dinov2_salad/models/dinov2/layers/patch_embed.py\\\': \\\'087f6d61b517ca6768817885dbcb4dcef7bf48540630f1e819784f6fc666d234\\\', \\\'models/dinov2_salad/models/dinov2/layers/swiglu_ffn.py\\\': \\\'615c5f2760f2d305c4a41d72a751744ab39af521b9b359542770dd01ac708df9\\\', \\\'models/dinov2_salad/models/dinov2/vision_transformer.py\\\': \\\'8000dbdb4983ffea6efaaacb18330edae8d8b45006e1706a616c2406b31d42d0\\\', \\\'models/dinov2_salad/models/helper.py\\\': \\\'ba09d0d82e14a61dc79e5cbe55085fd036f9fbccfa98c355d0523d5924a81c9d\\\', \\\'models/dinov2_salad/utils/__init__.py\\\': \\\'c83931f3656b48ebcdf43ffbe361d83377078d7f36bfd57a7be2ea8ffeb99350\\\', \\\'models/dinov2_salad/utils/losses.py\\\': \\\'e4026123698a0d79d8a80e5c5a972ea4eadb69494128d06fd0b6401572d8ef9b\\\', \\\'models/dinov2_salad/utils/validation.py\\\': \\\'36efdf67cd669414bef5deeaf8d03d4e1025d4a0db7c82c770d0292ab24547a0\\\', \\\'models/dinov2_salad/vpr_model.py\\\': \\\'dc8c6dcbb4a9d7f8118578f6f079ad1d457c119460858a8528a648d4ce8b66c0\\\', \\\'models/dirtorch/__init__.py\\\': \\\'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855\\\', \\\'models/dirtorch/model.py\\\': \\\'2c94b988aea04ae6f7d1d18e62d51d9725d3d69b202e0150b15b9bad1c8de922\\\', \\\'models/dirtorch/nets/__init__.py\\\': \\\'a0e90619d5d727adedb04ef236f868260a3fd3880c41369c45a2a399e72dc6ec\\\', \\\'models/dirtorch/nets/backbones/__init__.py\\\': \\\'823180b2d59d4610ad3e393f3f759fb414741c499c016ba62dddd78c99045b3a\\\', \\\'models/dirtorch/nets/backbones/resnet.py\\\': \\\'7dbed323e5becdb9a9ab231243c81de7b2f3caccd5733669f3afb8b7ce0e63a5\\\', \\\'models/dirtorch/nets/backbones/resnext101_features.py\\\': \\\'05beb400138540c98a8642b7e70791a0f6d3fe2f31208700ffa6e0aadf7126d5\\\', \\\'models/dirtorch/nets/layers/pooling.py\\\': \\\'99aae1883aee5e8281a7d43cc35289ee54f8a66349b3ddfa6e41bb9704ac49a6\\\', \\\'models/dirtorch/nets/rmac_resnet.py\\\': \\\'981253ebe70f347a6906cb7334f6b1cd9cd70fe2d46f2fa13657396fdde24a0a\\\', \\\'models/dirtorch/nets/rmac_resnet_fpn.py\\\': \\\'ebee584dc38af4cc5019d678079428468dab1000f7a41c8c9c59282207a4b6c3\\\', \\\'models/dirtorch/nets/rmac_resnext.py\\\': \\\'62a3bb8d9dcb8f971b92b700e5a75bfb8b4bc4bf1ea63299acf094e1f5ca2e3f\\\', \\\'models/dirtorch/utils/common.py\\\': \\\'5c1f6fafae5484ebc77305ce71244f076f41c570032a3198b738cc0b5865afdc\\\', \\\'models/dirtorch/utils/convenient.py\\\': \\\'e2de568214e9efdb68d38dd4fab2040b81cc93d6c2744194a3b261ba7f085a10\\\', \\\'models/dirtorch/utils/evaluation.py\\\': \\\'6363f73d114af2aed0cfaf37611f7f8ad9743ccc4c9e6162529062214f8c77c5\\\', \\\'models/dirtorch/utils/funcs.py\\\': \\\'01f2f245a1bc19e2aadaee1bc6e810ced6050f83596ae5cf5f35c95d2e8c30c8\\\', \\\'models/dirtorch/utils/pytorch_loader.py\\\': \\\'dfa87b8a3a292f4f3dd9ec6b79cc5fe5058ca0a1e3dfd77f66bb9a7e9832bf62\\\', \\\'models/dirtorch/utils/transforms.py\\\': \\\'9bb6f38c43af8939e3196664db84ccb3280f97257240cc40cec2e4d33fa4d9c1\\\', \\\'models/dirtorch/utils/transforms_tools.py\\\': \\\'57245fb2e0b83619db5c4f837f8bfd9b0391c412e6272a1c1c6131d60028ba05\\\', \\\'models/dkm/__init__.py\\\': \\\'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855\\\', \\\'models/dkm/models/__init__.py\\\': \\\'438d6932eeea9de5e0ea45d04eb6e761028180eeee79a0dc71178f8226d01e6d\\\', \\\'models/dkm/models/dkm.py\\\': \\\'f7bb80a1ff88e7dc92b656cede75ab2a5c1861b60b76473536a54f92a2aa4118\\\', \\\'models/dkm/models/encoders.py\\\': \\\'e8579cc5f734a2fcc911408b699b52e8c084e6ebbe9df8e868efb8f2c705760a\\\', \\\'models/dkm/models/model_zoo/DKMv3.py\\\': \\\'928a25a04916c29050a9a5523fc065ae84116075fb245dd4cadd3faa243d1187\\\', \\\'models/dkm/models/model_zoo/__init__.py\\\': \\\'d79d3badbf95d8fc10a577fec953ea275defdc38983173f96963fd477170e18a\\\', \\\'models/dkm/utils/__init__.py\\\': \\\'201a745886ff89866d38afc99260426ef04a4056828356dad78fd52d71a779ce\\\', \\\'models/dkm/utils/kde.py\\\': \\\'018a50faa554ae4b92bb7d37334f0ea53711ad57ceefe12d027b5ae7d22d28be\\\', \\\'models/dkm/utils/local_correlation.py\\\': \\\'51010bba5070ba7840d3ead78a2d57857067fde8bf1014c9b7741592d259001c\\\', \\\'models/dkm/utils/transforms.py\\\': \\\'2d433f92a4575a572c4b7907d06fd6e9d2479eb2f8dc2ce004e07e9a1386f182\\\', \\\'models/dkm/utils/utils.py\\\': \\\'1a3081ed481b17b21066c8fc44aaf3e6794180be7d2f352cbd861c2fe9a491fd\\\', \\\'models/doppelgangers/datasets/pairwise_disambiguation_dataset.py\\\': \\\'2fdfc043440c15de26ce297f75ab110a78325dbbdf2e9376f0b0c339b736e5b6\\\', \\\'models/doppelgangers/datasets/sfm_disambiguation_dataset.py\\\': \\\'958b315add979f7b6c7ffdfb53b1abe6b6b045f87c2a24040d1fcf707b302ad9\\\', \\\'models/doppelgangers/models/cnn_classifier.py\\\': \\\'8440b681d460f860c8690964b62a3ca1cadfd4787b562f77d17614a62e2cb494\\\', \\\'models/doppelgangers/third_party/loftr/__init__.py\\\': \\\'66d0f1572f4267b19af8bb6d21ff45cecc8c5467fe6f5761a0e8961cda19cb83\\\', \\\'models/doppelgangers/third_party/loftr/backbone/__init__.py\\\': \\\'50974a8fe95bff53753f40b2af21ec6269d3dad123ffeea6e026013658884f83\\\', \\\'models/doppelgangers/third_party/loftr/backbone/resnet_fpn.py\\\': \\\'88e63632386bd39c0f6b3ff5db0a5866ee99a5c6851747c287763d6e63fe77cb\\\', \\\'models/doppelgangers/third_party/loftr/loftr.py\\\': \\\'0452f4c18e6de17ffe8e51a55f09e6c59d8e35392ba7b075d0d60b4f49ef9455\\\', \\\'models/doppelgangers/third_party/loftr/loftr_module/__init__.py\\\': \\\'25fbc03b34ce660b9534db639a7a9b21f30c9450922671a91bb226ede6bce3b0\\\', \\\'models/doppelgangers/third_party/loftr/loftr_module/fine_preprocess.py\\\': \\\'1eeb83778a4c50b5752ff37d2b8ac72d76706ea4fe3bc222ab8c47263279fafe\\\', \\\'models/doppelgangers/third_party/loftr/loftr_module/linear_attention.py\\\': \\\'0bb92eeb20133ae75ad489c27f563a8a0973afeef48a5072098845684af04ef1\\\', \\\'models/doppelgangers/third_party/loftr/loftr_module/transformer.py\\\': \\\'9c24fcab3bfd655363b52f85e0108de1e08b4f619948fac3e8e6501537024610\\\', \\\'models/doppelgangers/third_party/loftr/utils/coarse_matching.py\\\': \\\'6a3dfc2f3387d1de7ddc7f9353f753ad4522f4c5d4143a539cbb9216b2c86147\\\', \\\'models/doppelgangers/third_party/loftr/utils/cvpr_ds_config.py\\\': \\\'30bc13adf6cb964952ba48efe7d96ba9d049f45c8bf82eb325d5bd3dbb00386b\\\', \\\'models/doppelgangers/third_party/loftr/utils/cvpr_ds_config_keypoint.py\\\': \\\'b9f52f6394cf60907a1b093c1e2924f045b5ff1bdcff935ab0dddb48ce1fe3fb\\\', \\\'models/doppelgangers/third_party/loftr/utils/cvpr_ds_config_keypoint_cp.py\\\': \\\'5be261dfec3e673d4e6aaf7194ae4f1854a360166810abd5a2c93a4b3a0995ce\\\', \\\'models/doppelgangers/third_party/loftr/utils/cvpr_ds_config_keypoint_fc.py\\\': \\\'1a28ff98a82002efbe48e25ab8df06d2d9023cef976a0b17976a3ce819df7f80\\\', \\\'models/doppelgangers/third_party/loftr/utils/cvpr_ds_config_keypoint_fpn.py\\\': \\\'d650e185b8f4bce905639ca58ffa4df9f5d8598413e2212e4395f5b900cfd195\\\', \\\'models/doppelgangers/third_party/loftr/utils/cvpr_ds_config_keypoint_pca.py\\\': \\\'8f830ab8d842e77c0b277a549cabf8e73035c05a445d8505e2249a0eba846e63\\\', \\\'models/doppelgangers/third_party/loftr/utils/fine_matching.py\\\': \\\'35a1203e7d0e8cf002770393ab0667f321c5cd6622df6805d80c85e98da5a0ef\\\', \\\'models/doppelgangers/third_party/loftr/utils/geometry.py\\\': \\\'d10afe85e7144777343ff193d16baab514683a19f207832422f15bc771226668\\\', \\\'models/doppelgangers/third_party/loftr/utils/position_encoding.py\\\': \\\'285f4a55ee46d2ec01166250b0a6ed36d7e584aa48094311734b6f2933979d91\\\', \\\'models/doppelgangers/third_party/loftr/utils/superglue.py\\\': \\\'b73d12cba49bc2aa6d21d0bdc69eb1f9f77bd966076a0b3b8c24c5ad83569f1d\\\', \\\'models/doppelgangers/third_party/loftr/utils/supervision.py\\\': \\\'890bf648c37e9a359a2ef864df8006a38b11014d722ca972d71367d0c20d0737\\\', \\\'models/doppelgangers/utils/database.py\\\': \\\'dcc0f37fcc4f584d9afb5a4f530dd1d7226dbc74eb9c1e4808bb1583f35d76bc\\\', \\\'models/doppelgangers/utils/dataset.py\\\': \\\'2734aff5479d290684ca6413462cf7bc420595f2dc8d2162c8adce931db5b84a\\\', \\\'models/doppelgangers/utils/loftr_matches.py\\\': \\\'fdcc05a0a97de63019cea745ba706387bb01a8728e37d74879b80c6201fce30c\\\', \\\'models/doppelgangers/utils/process_database.py\\\': \\\'5418e8109236bde0abc426e94f98666e037088c9c2df3c58bf707edbed5d1fe2\\\', \\\'models/ecotr/__init__.py\\\': \\\'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855\\\', \\\'models/ecotr/config/__init__.py\\\': \\\'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855\\\', \\\'models/ecotr/config/default.py\\\': \\\'502da286df40608d03d9c4af911fd97816b510db85323d6bf3a976ed52ca9616\\\', \\\'models/ecotr/models/__init__.py\\\': \\\'39df6d38a6209c8c8508490b04c6b8c77614bf4c281c532b19aff1cd45545e55\\\', \\\'models/ecotr/models/base/__init__.py\\\': \\\'2e3db21440c0460182b1de61830b160116713b777c84a14b59af492abeaa35b6\\\', \\\'models/ecotr/models/base/resnet.py\\\': \\\'a239b1e14eebf4d844d9d56442c341eb4ff74ee984d5913a53a32d25b83808fc\\\', \\\'models/ecotr/models/ecotr_engines.py\\\': \\\'df100a8bae058a110b756371a10fc720c7a8846cf73670192104cad0e90b1a78\\\', \\\'models/ecotr/models/ecotr_model.py\\\': \\\'8c358de59f37620a819d2fb73e6234b0737ad327658bda4b58b8d610f32e0881\\\', \\\'models/ecotr/models/ecotr_modules/__init__.py\\\': \\\'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855\\\', \\\'models/ecotr/models/ecotr_modules/ecotr_backbone.py\\\': \\\'9c4c2d78e3aff717e852f5f1b5e9436eb1f4f7bd9798b2e779891ab0ce54e7e9\\\', \\\'models/ecotr/models/ecotr_modules/misc.py\\\': \\\'0149bbeee4e4f23048fa04ef9ced65dacf1e208090229c0b314daf341eff6d52\\\', \\\'models/ecotr/models/ecotr_modules/position_encoding.py\\\': \\\'dca3f7b610c36648a4877531eb2197413c0ce60c05f5de91f114c7b04dee59b8\\\', \\\'models/ecotr/models/ecotr_modules/query_clustering.py\\\': \\\'fd64f63686ae4874bc18e195e50949ee8e30c546df11b7f3935a770153ae216f\\\', \\\'models/ecotr/models/ecotr_modules/transformer.py\\\': \\\'984d3eab0483194dbc28aabf9d685a37af46781c0bb3df24ca83efa0321c976e\\\', \\\'models/ecotr/models/utils/__init__.py\\\': \\\'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855\\\', \\\'models/ecotr/models/utils/inference_helper.py\\\': \\\'8aac22dcb62babe54a75b26b06b4959061de36ee5eb0581d1b3b59c9a1f3cde0\\\', \\\'models/ecotr/models/utils/utils.py\\\': \\\'2097d355e173d51bd8c0a8a1b5dbd5825afcc42f44b1c10b8a3337c37e820cb9\\\', \\\'models/ecotr/utils/__init__.py\\\': \\\'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855\\\', \\\'models/ecotr/utils/misc.py\\\': \\\'76a3ae3ce6fb18bb0e994574eb18408055e3da24605cb07f7de90eff6608e311\\\', \\\'models/edgepoint2/edgepoint2.py\\\': \\\'4805b73764338baf384bb18f8ebd99fe0a5dc371a6ffbbcd466e33d0bff23f95\\\', \\\'models/edgepoint2/modules.py\\\': \\\'4d41688cfb3eb907c502ba343ec6f59219f3e58ac6cae370fea5ca095387230e\\\', \\\'models/efficientloftr/config/default.py\\\': \\\'f8962a063191b40cdfc16d038e2517bfa5993b702e4d5df1ab81091b2e73a7e3\\\', \\\'models/efficientloftr/loftr/__init__.py\\\': \\\'f5ddb56f36c034d2933d12fcaf1f8c74580a6903df2a65531eb00c79d085de27\\\', \\\'models/efficientloftr/loftr/backbone/__init__.py\\\': \\\'6411d463dfb84593950be726de63cd8bf9dae8dac3f8955afceb9725d5da471e\\\', \\\'models/efficientloftr/loftr/backbone/backbone.py\\\': \\\'34fa7ee35cc808297ac218f51f03a302896e82938081a1561a02366322340c59\\\', \\\'models/efficientloftr/loftr/backbone/repvgg.py\\\': \\\'6f135d5eeb77210a8581e3997a8c2862b22087d97249a4031353b01857bed49d\\\', \\\'models/efficientloftr/loftr/loftr.py\\\': \\\'72a6fa0629c9d44418c0250ad1e997953e993dee01778c4f60d8d462b8ce9e0f\\\', \\\'models/efficientloftr/loftr/loftr_module/__init__.py\\\': \\\'25fbc03b34ce660b9534db639a7a9b21f30c9450922671a91bb226ede6bce3b0\\\', \\\'models/efficientloftr/loftr/loftr_module/fine_preprocess.py\\\': \\\'5ebffc86a6276af3f6089b014973efe324655460744186c9eedda8807775056c\\\', \\\'models/efficientloftr/loftr/loftr_module/linear_attention.py\\\': \\\'574ca52f754146e4025bb9d321f9bef971013fb56b31ab5ad3cc6892988c7e23\\\', \\\'models/efficientloftr/loftr/loftr_module/transformer.py\\\': \\\'424e05d086934208fe354d6475e83096abf31e582cf1e1e06581626b8263e488\\\', \\\'models/efficientloftr/loftr/utils/coarse_matching.py\\\': \\\'447a8b50206367d45ceda3f08410641c33fc9d1eb07f2fe07aea4f1bd60e4fa4\\\', \\\'models/efficientloftr/loftr/utils/fine_matching.py\\\': \\\'f0bc0c740ac05b97bf57eed4d6614b022db135929b2a398bc28d39f8beaf7402\\\', \\\'models/efficientloftr/loftr/utils/full_config.py\\\': \\\'325e4d24aa701199264b310c77233398e7d37aff53881b33c0c1efa2b76525e3\\\', \\\'models/efficientloftr/loftr/utils/geometry.py\\\': \\\'d10afe85e7144777343ff193d16baab514683a19f207832422f15bc771226668\\\', \\\'models/efficientloftr/loftr/utils/opt_config.py\\\': \\\'85b3be371d2c1949a81f035168b5f519382f4772e444280d1c97a6283a6d7216\\\', \\\'models/efficientloftr/loftr/utils/position_encoding.py\\\': \\\'6a7a9471e44366f248721a1a12b99e50291d53850e374d56e8f598f9cdd110b3\\\', \\\'models/featurebooster/__init__.py\\\': \\\'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855\\\', \\\'models/featurebooster/model.py\\\': \\\'8c5c6d7c7a5f6c3d9f181384b7df9037a80cbda50cdcaa8a270636053ea65a6e\\\', \\\'models/fftformer/fftformer_arch.py\\\': \\\'560b0bd026a9dec0b3f2d320392818d1a8f1e54a35a6cbe473e951338cc03853\\\', \\\'models/fsnet/FSNet_model.py\\\': \\\'336f5fb81f9e91ff0e9c1e62a97ed64322635a64ec1a7c051d97f3a506bf1f5d\\\', \\\'models/fsnet/modules/backbone.py\\\': \\\'6d5ef6197e69fa6de2de40c451de03bcc707af06c9438cb81e966b058eeb06a4\\\', \\\'models/fsnet/modules/epipolar_sampling.py\\\': \\\'f2a54846ad861b4adbb5c7988b48e3234cb3620862a3e14308d76ba1467f96db\\\', \\\'models/fsnet/modules/error_regressor.py\\\': \\\'6ac2ebc9f2fe746b52a0aab4ebaff77433fddcd3394dca2605421abd1b2b96b1\\\', \\\'models/fsnet/modules/linear_attention.py\\\': \\\'409a8350e812a1a11c3f9d271d4b150c4ce314e43a084f0fd557b04b46585c67\\\', \\\'models/fsnet/modules/regressor_layers.py\\\': \\\'ab32c48d6545ff590cbdaa3c3697aa9dae9b899ab93421b48a9e0755ddc5eeaf\\\', \\\'models/fsnet/modules/transformer.py\\\': \\\'a09c13dee63b8d9f5dfb0e79324134b7033bfef019252e40eac2a09b23476dfc\\\', \\\'models/fsnet/modules/utils.py\\\': \\\'6c6aa835b6c4c03f58a6d9b4cfa5b606530aa583477561ae72cf62b6f67a3b1c\\\', \\\'models/gim/dkm/__init__.py\\\': \\\'063ca04c7ec75e15dde3c45a10320b16615c1a4ec3e05fac313de196e05787b5\\\', \\\'models/gim/dkm/benchmarks/__init__.py\\\': \\\'691adcef62ab72188c927e033e02f7cfc11755db963855c3fd69ce318e76759d\\\', \\\'models/gim/dkm/benchmarks/hpatches_sequences_homog_benchmark.py\\\': \\\'2705fb56811ade53b221ea1fe3e162f6d33701f2ade943b48adec593f9d05a87\\\', \\\'models/gim/dkm/benchmarks/megadepth1500_benchmark.py\\\': \\\'1f248be12cd3b0455869a9e4d10aa1b757fbd549ac9bc0f4ecce4caa1b946357\\\', \\\'models/gim/dkm/benchmarks/megadepth_dense_benchmark.py\\\': \\\'f8753d11bde7f981a5d24eac7ce56893a841ef5f46b479f89fc99e52f49f3ed5\\\', \\\'models/gim/dkm/benchmarks/scannet_benchmark.py\\\': \\\'12b394db1647e619b6a7c7ea49748d14523d1a84faf947bfc6fc88e4a02c0a90\\\', \\\'models/gim/dkm/checkpointing/__init__.py\\\': \\\'76d5aa36ec12d9aab0b9cff5f14849c516743edf4c873a01421cb87bc0d06742\\\', \\\'models/gim/dkm/checkpointing/checkpoint.py\\\': \\\'9ea8b36c035c962ef21231035e3c61161c926bfe673e23bedb993ba01a7233a1\\\', \\\'models/gim/dkm/datasets/__init__.py\\\': \\\'6a1f0acd0dc59fbb22157a6b41174abf720b87c495d55998ad073ba898589e2d\\\', \\\'models/gim/dkm/datasets/megadepth.py\\\': \\\'f693a8242f2df7ece588ba1ad665196c2786136760684923855bbb37b61d5462\\\', \\\'models/gim/dkm/datasets/scannet.py\\\': \\\'f06f81ea26a50c7c563bd30d9a0a4530e9ddcf81100fcc3c193132e514921b28\\\', \\\'models/gim/dkm/losses/__init__.py\\\': \\\'92cfb9a95652d1245195c24e84c9be8db31492dcc55a7a09004cb19e63a19e10\\\', \\\'models/gim/dkm/losses/depth_match_regression_loss.py\\\': \\\'15d9868ff48c2dce9d9748a79294b007805f51d0e04feabdc46bf8cebaf3783e\\\', \\\'models/gim/dkm/models/__init__.py\\\': \\\'438d6932eeea9de5e0ea45d04eb6e761028180eeee79a0dc71178f8226d01e6d\\\', \\\'models/gim/dkm/models/dkm.py\\\': \\\'8c16cba67a21ac9ba0c8c8d79e109ce8ffef92c8247ac4155d18d5ef7e50ed31\\\', \\\'models/gim/dkm/models/encoders.py\\\': \\\'bdb05164d991f0e09dc7c96f3789c3dfa352d0010bd4ac37fbd615fa2f54f420\\\', \\\'models/gim/dkm/models/model_zoo/DKMv3.py\\\': \\\'0c947a9305ad16a9a1e881d976a5b8fab4b16586d4ad0121df481abc8415fdac\\\', \\\'models/gim/dkm/models/model_zoo/__init__.py\\\': \\\'157f585786f0db07a14b694f12fe1f34eda9eddc53b6f2c0be2a446473d82df4\\\', \\\'models/gim/dkm/train/__init__.py\\\': \\\'39c45fbe55e7d94aa69318dec2e6eceffa493847e36ce9850417c23959bb9afd\\\', \\\'models/gim/dkm/train/train.py\\\': \\\'5da4bd6278b8498cdf41c45e2a45e5b128e6782af65703dc1f76bd5a5ec63309\\\', \\\'models/gim/dkm/utils/__init__.py\\\': \\\'201a745886ff89866d38afc99260426ef04a4056828356dad78fd52d71a779ce\\\', \\\'models/gim/dkm/utils/kde.py\\\': \\\'018a50faa554ae4b92bb7d37334f0ea53711ad57ceefe12d027b5ae7d22d28be\\\', \\\'models/gim/dkm/utils/local_correlation.py\\\': \\\'51010bba5070ba7840d3ead78a2d57857067fde8bf1014c9b7741592d259001c\\\', \\\'models/gim/dkm/utils/transforms.py\\\': \\\'2d433f92a4575a572c4b7907d06fd6e9d2479eb2f8dc2ce004e07e9a1386f182\\\', \\\'models/gim/dkm/utils/utils.py\\\': \\\'cf6a86e1a45f20b83c2c3cec0967d9c77753a24e44862433f3c69390ffca9d3c\\\', \\\'models/gim/gluefactory/__init__.py\\\': \\\'36070d930e91743bd34c47e81f2425490f06273bc9e038217d619bd7c0a9037a\\\', \\\'models/gim/gluefactory/datasets/__init__.py\\\': \\\'7942df2ec262094f959f871e7f8de3400f74a340d8ba83ec7ab114c3b1a2ae80\\\', \\\'models/gim/gluefactory/datasets/augmentations.py\\\': \\\'86c9c0297c4c5391ea54fa703856e1d04079306115252d0834058160e939b661\\\', \\\'models/gim/gluefactory/datasets/base_dataset.py\\\': \\\'b818cb72ce2d14ad9337728756292feff9d219e25d249d4e36e38dcc85838094\\\', \\\'models/gim/gluefactory/datasets/eth3d.py\\\': \\\'e0b2783c8b4d756b98750450631bd46d8a5130ca596a8463ac10b0d7099f8bfc\\\', \\\'models/gim/gluefactory/datasets/homographies.py\\\': \\\'d86b45e3648c7c39df51ea59a8dae6959135d787fb08d937a0d450716b6e0657\\\', \\\'models/gim/gluefactory/datasets/hpatches.py\\\': \\\'664955840b07e291af3a157cae68ea13aa4ecfdffb74f955e61b552ea1dab09f\\\', \\\'models/gim/gluefactory/datasets/image_folder.py\\\': \\\'d5af2bd9f21ff998b30ac1c69a09a47f2a888305af27375e58091363dc20a037\\\', \\\'models/gim/gluefactory/datasets/image_pairs.py\\\': \\\'d74a1963ad658c88a30c55b4038cad5c9456a19bedc9d3fae4086050c97af8fd\\\', \\\'models/gim/gluefactory/datasets/megadepth.py\\\': \\\'f9769a4708650404039401ea3d4e74813c48f4ac4dbe7905b1c6f77e3420b4ca\\\', \\\'models/gim/gluefactory/datasets/utils.py\\\': \\\'bf560f03ac096df826b1f35d35427e2f23c29578207edfab649f72e55ab7e574\\\', \\\'models/gim/gluefactory/eval/__init__.py\\\': \\\'b3d5d3f0b93bc36b0048b013efd91319d612a242d069435a35aaee88be1f57a0\\\', \\\'models/gim/gluefactory/eval/eth3d.py\\\': \\\'d15a6643885d601afe7189b7449e495def11bd5a24781eef1c228365f47629e0\\\', \\\'models/gim/gluefactory/eval/eval_pipeline.py\\\': \\\'602deb908fafd91d8a8fddb4197838ed0739f7abc2a1e82c599fa57843f80cfd\\\', \\\'models/gim/gluefactory/eval/hpatches.py\\\': \\\'9378ced52abdd9ea92d3e3fd249a15f95cd3df910e576afa57bfe28fc87249f8\\\', \\\'models/gim/gluefactory/eval/inspect.py\\\': \\\'6ecd33f735b9cf9cccf9012ae0377e74813c37eb7c387940a332d11a13b6ee11\\\', \\\'models/gim/gluefactory/eval/io.py\\\': \\\'b9e02052a496bde7d82c6523f1556bc741f234961d1da9fb19afb40cd8d18b1b\\\', \\\'models/gim/gluefactory/eval/megadepth1500.py\\\': \\\'19f58daf07a562dd48ed12ecab0ead8e24bca28fab315a7b4761fb1d52db7964\\\', \\\'models/gim/gluefactory/eval/utils.py\\\': \\\'88542d86bbc847b5c61bfb154f3f4fc788c04c4512223f84dad4a3f02d5a1deb\\\', \\\'models/gim/gluefactory/geometry/depth.py\\\': \\\'6c6b5644997111a794a3c9f62fea99a4a899fac9d5dbfc6042518f77f794b875\\\', \\\'models/gim/gluefactory/geometry/epipolar.py\\\': \\\'c9210d1a00ec3ca62899948c33ee8532a4a6c28c25424d5f883c1105e107509f\\\', \\\'models/gim/gluefactory/geometry/gt_generation.py\\\': \\\'6e93a642a892964a31fee79d074578ad32435786202fc51b62e3f76456918d7a\\\', \\\'models/gim/gluefactory/geometry/homography.py\\\': \\\'e1a3fee3b6ebad215c8903777e3bb6aa98d58e2f51d64d8e4c3a155fd0a4e17d\\\', \\\'models/gim/gluefactory/geometry/utils.py\\\': \\\'431a608429fdf01eee3291e62bfdb82a46d842a76e3a0f4b39b667b6e607e2f9\\\', \\\'models/gim/gluefactory/geometry/wrappers.py\\\': \\\'24b35bbe44936c1d5edeca0ff8520311961ecd125f12fc98519c2a50fe92e39f\\\', \\\'models/gim/gluefactory/models/__init__.py\\\': \\\'843b7a7af3830756eeacc75eb536479b5d8168e9c141bbbf7505e5e9dc2d5f14\\\', \\\'models/gim/gluefactory/models/backbones/__init__.py\\\': \\\'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855\\\', \\\'models/gim/gluefactory/models/backbones/dinov2.py\\\': \\\'b02967ae8c8f94e149c9b9b3c765f5ebad00e7ec62b53f3f9080d97294568920\\\', \\\'models/gim/gluefactory/models/base_model.py\\\': \\\'8c12ce9372daa374aae97546680839831d044c72d2bf5c7f60e2fdc5ea116476\\\', \\\'models/gim/gluefactory/models/cache_loader.py\\\': \\\'1c94feb370e5cd27177eccca977d66290c624eed12c63cbf94f8b89d710a2cf7\\\', \\\'models/gim/gluefactory/models/extractors/__init__.py\\\': \\\'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855\\\', \\\'models/gim/gluefactory/models/extractors/aliked.py\\\': \\\'3ecb33096307c58f064a5443a9d7564188d09daa9105e624f39f8b769d11c1e4\\\', \\\'models/gim/gluefactory/models/extractors/disk_kornia.py\\\': \\\'d957ce934416f5df29b00787d71520dc659c6a7ade03f3ac9cadb52f8edbc99e\\\', \\\'models/gim/gluefactory/models/extractors/grid_extractor.py\\\': \\\'6386bc5f4803ee6e81806ace6a7cbaa2621a81a6c99f0507f58792c480c8be09\\\', \\\'models/gim/gluefactory/models/extractors/keynet_affnet_hardnet.py\\\': \\\'941d67c634299d041ba0efe574f398c881a3f608837b367460f0ee1f88799142\\\', \\\'models/gim/gluefactory/models/extractors/mixed.py\\\': \\\'0bb8afbdaf972e1e5d8a714a1d4e7af4a257e427560d9c52b20cc707a52500af\\\', \\\'models/gim/gluefactory/models/extractors/sift.py\\\': \\\'14234d9c24362d4c8c5663819ac8cef5ed2d18e9d01ab151bc4a57a37cd3296b\\\', \\\'models/gim/gluefactory/models/extractors/sift_kornia.py\\\': \\\'ac2b9ca40511bf5d1ccfe7dfdf1130d166a0ef0aea5b92ea99c141232971ac24\\\', \\\'models/gim/gluefactory/models/extractors/superpoint_open.py\\\': \\\'99a6ead879f6229da81eaeba3876b4f665a03b84e88ca580d09bbfbfd94e20fc\\\', \\\'models/gim/gluefactory/models/lines/__init__.py\\\': \\\'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855\\\', \\\'models/gim/gluefactory/models/lines/deeplsd.py\\\': \\\'de0d9befc816b2844090112a1e62fbb736f3687fcae90f447197e0eff25e8ddf\\\', \\\'models/gim/gluefactory/models/lines/lsd.py\\\': \\\'dac8d9e1cb9c704784a41fed4517933bd44ad76d0bf20d596847e4256fa3521c\\\', \\\'models/gim/gluefactory/models/lines/wireframe.py\\\': \\\'131a692a536dd8ac5b427dadef4167eb822d2bb12b4c8a4eb314f1cd530325cd\\\', \\\'models/gim/gluefactory/models/matchers/__init__.py\\\': \\\'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855\\\', \\\'models/gim/gluefactory/models/matchers/adalam.py\\\': \\\'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855\\\', \\\'models/gim/gluefactory/models/matchers/depth_matcher.py\\\': \\\'434dba271a4c107ba2f009fbcbf9d238edc5cac550e4f09d3e003021e027b055\\\', \\\'models/gim/gluefactory/models/matchers/gluestick.py\\\': \\\'7af8a836f89822d4d2e5ef92e3611ee7f3ea6108924d79c9897f310b92e78996\\\', \\\'models/gim/gluefactory/models/matchers/homography_matcher.py\\\': \\\'3d4dc3d240d32d6999538e7e5cd1295562d7cc22f9ec1fe87f0e70f1284e34a2\\\', \\\'models/gim/gluefactory/models/matchers/kornia_loftr.py\\\': \\\'af382580dbeaa374143bd4ee7c7b17f005734b8feb7699707941b2f657c83b66\\\', \\\'models/gim/gluefactory/models/matchers/lightglue.py\\\': \\\'a81c04b3e3a0f5f3df61c4f1e379a15e110bee2d072e4662436e18b4d44b20c6\\\', \\\'models/gim/gluefactory/models/matchers/lightglue_pretrained.py\\\': \\\'24d814aa32110f7c48ffaf0cc004ec6ccae7bb6e150252a96d7e41830ac09cca\\\', \\\'models/gim/gluefactory/models/matchers/nearest_neighbor_matcher.py\\\': \\\'921d86bcf6362518b8af3e261314617191e4077310c00b13836fa3bd662e96f7\\\', \\\'models/gim/gluefactory/models/triplet_pipeline.py\\\': \\\'668a83863e434ccb3d463b9a35b7da001c15cede035ad7b793c650dad2c3cbc8\\\', \\\'models/gim/gluefactory/models/two_view_pipeline.py\\\': \\\'b6b25a53ebc57e4a385a3a26520336ec8512b02b93f360497e95e92649f5ac37\\\', \\\'models/gim/gluefactory/models/utils/__init__.py\\\': \\\'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855\\\', \\\'models/gim/gluefactory/models/utils/losses.py\\\': \\\'f20a606f692f2122e56c77f232de77a6ad1cbe3add13bb2a6e840c17b7bbef35\\\', \\\'models/gim/gluefactory/models/utils/metrics.py\\\': \\\'3094a6d38c49bc5fd648a2f4308002206d4b9892cbf3ce59cbd51fef7d5d54e2\\\', \\\'models/gim/gluefactory/models/utils/misc.py\\\': \\\'d7945d745720d87b6714150a661ede09b3e608e2b8635f8a820fdafb9c4b14e6\\\', \\\'models/gim/gluefactory/robust_estimators/__init__.py\\\': \\\'23f18c47b089c8492f06a20cc52ecf26db11d0f98e05e0f71acbb9d7bbdb2aea\\\', \\\'models/gim/gluefactory/robust_estimators/base_estimator.py\\\': \\\'c2620099628801be85e7ea4ed1a8425b000f5a5247b9d6e569a38846095bab48\\\', \\\'models/gim/gluefactory/robust_estimators/homography/__init__.py\\\': \\\'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855\\\', \\\'models/gim/gluefactory/robust_estimators/homography/homography_est.py\\\': \\\'62dd25f8b46389b58a1e6a34f2b87b0716e5a512c81a9fad0e55a24b1cffc95e\\\', \\\'models/gim/gluefactory/robust_estimators/homography/opencv.py\\\': \\\'bb15be426c006087ac7e95a8ba393172f18d43181b456557f029ebc28762ee4c\\\', \\\'models/gim/gluefactory/robust_estimators/homography/poselib.py\\\': \\\'f7e6e66871114df6e92acb5b0e7f38d9f336a33b74bafe4e550201bd0f617198\\\', \\\'models/gim/gluefactory/robust_estimators/relative_pose/__init__.py\\\': \\\'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855\\\', \\\'models/gim/gluefactory/robust_estimators/relative_pose/opencv.py\\\': \\\'c091bc085b3efbf466659b93ac8865b860044f6d851c8b45eab56d379055ca84\\\', \\\'models/gim/gluefactory/robust_estimators/relative_pose/poselib.py\\\': \\\'b720bfcb1d0e206d65426a41f212fdf0078ccdc94419df663bddbe907d65ed52\\\', \\\'models/gim/gluefactory/robust_estimators/relative_pose/pycolmap.py\\\': \\\'fb88fb2152adca3f3da41a412e9b4259ae2a9ce7b1445a33b721efd3bdff980a\\\', \\\'models/gim/gluefactory/settings.py\\\': \\\'9bcc8412490fc010d23ff5f22559f86b05f3536b810774c4a6a57a89040d77fd\\\', \\\'models/gim/gluefactory/superpoint.py\\\': \\\'85f52379fde421012383db5ec3480e80ee34c1c572c16fc8d71117770d0ddf48\\\', \\\'models/gim/gluefactory/train.py\\\': \\\'02b4b35678a51f3c861d5299032444ad064cf07167b5e9a4e2588957b4af9e10\\\', \\\'models/gim/gluefactory/utils/__init__.py\\\': \\\'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855\\\', \\\'models/gim/gluefactory/utils/benchmark.py\\\': \\\'214e3a2982bf9a625bcf18ac29ecb59d4f4804a526cf335c6714201dc9c3e6a3\\\', \\\'models/gim/gluefactory/utils/experiments.py\\\': \\\'8745a257398c309f82bca38f168d434eddd968c7ba6dc03601eb574c32344ea2\\\', \\\'models/gim/gluefactory/utils/export_predictions.py\\\': \\\'eca912db5fc364b56cb8257015ba891f4c9c00475b5d84778594f2e37775755c\\\', \\\'models/gim/gluefactory/utils/image.py\\\': \\\'0d935921aea1f7d9098530fa1949c9538c2435955eddcad36adc4d91ec7ac89e\\\', \\\'models/gim/gluefactory/utils/misc.py\\\': \\\'cf1a75640626523d3d3bd8634a55cfb9e0988fcdd729b4c968b1613a5d26f30d\\\', \\\'models/gim/gluefactory/utils/patches.py\\\': \\\'9b8c321afc33410117693a2dbfebe78256be35481f6f20a0972505e172e68922\\\', \\\'models/gim/gluefactory/utils/stdout_capturing.py\\\': \\\'343a0e061892a9a722e74365f16cb11d8b16240b0a1942551c1af0adb4423fb1\\\', \\\'models/gim/gluefactory/utils/tensor.py\\\': \\\'bbe6fef5fed0c1e5c9c7fbeb70191bfdf4c0a99930e237c1e487349a1cfa7828\\\', \\\'models/gim/gluefactory/utils/tools.py\\\': \\\'92aab8971a65c58c1b1c3e8901d7f897a3643de653a8ac5be7c6df54c4509a18\\\', \\\'models/gim/gluefactory/visualization/global_frame.py\\\': \\\'e4c0d1933263db9d130af18fbe1da9008218c8d85c4e0d73367eaa1cf6c903ba\\\', \\\'models/gim/gluefactory/visualization/tools.py\\\': \\\'c1fef7901ab71cad65b765742e8ce25852e5ca32d41b2ba2cbbfce88a1aa1b14\\\', \\\'models/gim/gluefactory/visualization/two_view_frame.py\\\': \\\'25fd11143a8e38195abf794ad44bacc2bb65d1f39dc33af68d3c2d7f8514f28c\\\', \\\'models/gim/gluefactory/visualization/visualize_batch.py\\\': \\\'437c6bb534b3ef927b03dbc3ac2e6807ca1268bdc1cd438ef9ee8e0eac348bcb\\\', \\\'models/gim/gluefactory/visualization/viz2d.py\\\': \\\'6df61fbaa9c25dc296b1d8e4f1cf2138658e3283163b47991afb8c46230f596b\\\', \\\'models/gim/loftr/__init__.py\\\': \\\'430925a7f18c707aa6ad3b338beee519153188a67b421058276a2437fb91fe82\\\', \\\'models/gim/loftr/backbone/__init__.py\\\': \\\'6b3bc6c41d58d2381b4bf33176eff2bef44ffe87bfb2b79baf94548bcd630713\\\', \\\'models/gim/loftr/backbone/resnet.py\\\': \\\'1c9a4887bca6258d1140a4f95eb513663910ef47b48ee8e930cb1ca710b11f02\\\', \\\'models/gim/loftr/backbone/resnet_fpn.py\\\': \\\'a99820325e5056d47405389cadbbed6b40e3742c026766f6da3f6212dbc17431\\\', \\\'models/gim/loftr/config.py\\\': \\\'4ab01396375241e4dfc9f3573e392af031309807a1d9b83cf8f8ebf5c1461b5f\\\', \\\'models/gim/loftr/configs/__init__.py\\\': \\\'430925a7f18c707aa6ad3b338beee519153188a67b421058276a2437fb91fe82\\\', \\\'models/gim/loftr/configs/outdoor/__init__.py\\\': \\\'74b9f18fa288966a8820ed14c5d2fbcc892cd193fa0a52b674194f82da340dcc\\\', \\\'models/gim/loftr/loftr.py\\\': \\\'0e805ff979d05be3b628fb4ad1829cae9838cacd532a2db57ed631ba872cf867\\\', \\\'models/gim/loftr/misc.py\\\': \\\'5b98577aa86acc09f6112b0b4634152a59faed6b99f74d012bfd8d6702b88474\\\', \\\'models/gim/loftr/submodules/__init__.py\\\': \\\'25fbc03b34ce660b9534db639a7a9b21f30c9450922671a91bb226ede6bce3b0\\\', \\\'models/gim/loftr/submodules/attentions.py\\\': \\\'0bb92eeb20133ae75ad489c27f563a8a0973afeef48a5072098845684af04ef1\\\', \\\'models/gim/loftr/submodules/fine_preprocess.py\\\': \\\'1eeb83778a4c50b5752ff37d2b8ac72d76706ea4fe3bc222ab8c47263279fafe\\\', \\\'models/gim/loftr/submodules/transformer.py\\\': \\\'709717c3dd088da049475edfbc27361f48e3e9d7cfc8250978c2b0b29f3f513d\\\', \\\'models/gim/loftr/utils/__init__.py\\\': \\\'430925a7f18c707aa6ad3b338beee519153188a67b421058276a2437fb91fe82\\\', \\\'models/gim/loftr/utils/coarse_matching.py\\\': \\\'420a30d9297ec3bf83467895af094f7a3fc56c97101a65807bbd4bffa2a82c6f\\\', \\\'models/gim/loftr/utils/fine_matching.py\\\': \\\'88ab3f858a87c103bd71f8ebae8ea6215852ae9050f276ddb379e39756aef1dd\\\', \\\'models/gim/loftr/utils/position_encoding.py\\\': \\\'52909177098ccec371dc3258a3fe231ada1301b87200c40c631039c21f53d60a\\\', \\\'models/grounded_sam/__init__.py\\\': \\\'3bfd83fd743a20da9bb3b4cde817a26cc4637a5ec477309a4489fe4e49abe298\\\', \\\'models/grounded_sam/model.py\\\': \\\'96fdbca4f2e127d05c639ddf2f51afa8f65f4898fc52bf1df4e61904f0d9a16d\\\', \\\'models/isc/model.py\\\': \\\'3508af8ff66ff94196f1ad3289ca71577b5598af85f03b5df6e4046904ee64d8\\\', \\\'models/lanet/__init__.py\\\': \\\'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855\\\', \\\'models/lanet/network_v1/__init__.py\\\': \\\'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855\\\', \\\'models/lanet/network_v1/model.py\\\': \\\'54091a84f648eb5612d826dfac3afda7eef8df561f38dcfb2a8d4f6d7949b1b1\\\', \\\'models/lanet/network_v1/modules.py\\\': \\\'c48dfd7f93c503fe7be7dbe0811bb71c452b23756290d3943461735425bfb926\\\', \\\'models/lanet/utils.py\\\': \\\'0ffe977c50b185a80d15afc735892e2a6ffa8e00979cfaa8dc3fad73a6e9ffdc\\\', \\\'models/mapfree/pose_solver.py\\\': \\\'e9ad7b3a8d991e9c23d36941bf18b503b53fc06d9ea056568f61c7696907f31f\\\', \\\'models/mast3r/matching.py\\\': \\\'01a1a3d7a4ec102f3084f5d0fb9de557999d23da88dc298ac9ba2c10e930bab9\\\', \\\'models/mast3r/model.py\\\': \\\'a403a1ea8039af7b001567b1f5bba71cf32a1d69ef0fb18db7030aa69685848f\\\', \\\'models/mast3r/visloc.py\\\': \\\'32044430af068dec395f00762006ce15ff801e254f14f5fb9a3ff8c9d361a8c8\\\', \\\'models/mast3r/visloc_utils.py\\\': \\\'ae4e4dd28d449abcb1b97c8a948e41d4bf41866ada893321cb0a7176bbe36821\\\', \\\'models/matchformer/__init__.py\\\': \\\'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855\\\', \\\'models/matchformer/backbone/__init__.py\\\': \\\'aef93782179bcc18fa3de70d9ae299b8365fa3f6a99caa5ab80ff54a0b9d240d\\\', \\\'models/matchformer/backbone/coarse_matching.py\\\': \\\'07c6c77adefaf985432625f1cfefebcbb7fc8dd26ef1a07888ea148c684607e8\\\', \\\'models/matchformer/backbone/fine_matching.py\\\': \\\'823dd17053a8559f7b1ca32e41b46552c587c295ae1ab988ad75d2c91448cf3a\\\', \\\'models/matchformer/backbone/fine_preprocess.py\\\': \\\'1eeb83778a4c50b5752ff37d2b8ac72d76706ea4fe3bc222ab8c47263279fafe\\\', \\\'models/matchformer/backbone/match_LA_large.py\\\': \\\'c4e1d42f1d199ed2a9bb113c1dd1a5e4ce24cdfa78f684b63e505fd95d3dd507\\\', \\\'models/matchformer/backbone/match_LA_lite.py\\\': \\\'7837534c7a1de60cb914818e56089832f52f10ee5d2744ea776aef61260885f8\\\', \\\'models/matchformer/backbone/match_SEA_large.py\\\': \\\'719cfde5d44854240a555758a3da98dc28a48f45fba8b268781a1d44cf25f975\\\', \\\'models/matchformer/backbone/match_SEA_lite.py\\\': \\\'4b6b3ec96af6a5112b9118ca02717887fa3de41737c47e3d613817f7d667e9a1\\\', \\\'models/matchformer/data.py\\\': \\\'5e070a42f9294a085435c406cb835b0be84369fe34f2cf5ad42d35c85a06ba18\\\', \\\'models/matchformer/defaultmf.py\\\': \\\'94f90447e0dcc8debd39afee50682c39b4b183064e802c12ab698ae011a1ea27\\\', \\\'models/matchformer/lightning_loftr.py\\\': \\\'79a89c857b1f8a29dd7610c08879150c309d29381296a6fb920ed9a3fbc2e9a2\\\', \\\'models/matchformer/matchformer.py\\\': \\\'35a237de23838daa7798b7f553e972f8cd8a2795b6e51a4bb4eeafd59e5a0c42\\\', \\\'models/matchformer/utils/augment.py\\\': \\\'0faf76ee3f598c8e1973178dc203a4342b82db838ee3634795f63324726e6ac3\\\', \\\'models/matchformer/utils/comm.py\\\': \\\'b5a6eb8e91d95c946337b5657010abc76044a7ed0e8647f5171b8c5d0418b2ea\\\', \\\'models/matchformer/utils/dataloader.py\\\': \\\'beeed315d335b37302d20cfe6beb9115cebdd796691d701a7c062d204be42730\\\', \\\'models/matchformer/utils/metrics.py\\\': \\\'d510d8ce11fa1543dce698eaa590dc6b70ac2f7ace5d67590d3572730715270d\\\', \\\'models/matchformer/utils/misc.py\\\': \\\'c3ecbb78e9b7606182955d81e95fb5a6be3108f4fe8c9fc2addb6836af3bfb04\\\', \\\'models/matchformer/utils/profiler.py\\\': \\\'e2d183e0be8c6edb168270be3eb1dfe6e5f56b4e8ca569b3ce16f574c78dbcf8\\\', \\\'models/metric3d/model.py\\\': \\\'a3d22f5b78b134babad8f9925cd2e89033c2429ceea9b816963b732e564f9038\\\', \\\'models/mickey/benchmarks/reprojection.py\\\': \\\'c36806ffe8fcaeda96cce4feb35d63a3d1c151dfd3721f14a50c486aa01be32b\\\', \\\'models/mickey/benchmarks/utils.py\\\': \\\'f7f0f682f4e2f2fd6014d835ad3707bf2f05132d9ec9c631536543879dec48d3\\\', \\\'models/mickey/config/default.py\\\': \\\'f8184c126f1efaaf76876f7adc25f241580841b574d02562d149680577f82924\\\', \\\'models/mickey/datasets/datamodules.py\\\': \\\'6ce4601eef9c47e1ab4cf5d18a34094ae7ed0ebef4e91f94f105a74ccf8f3b62\\\', \\\'models/mickey/datasets/mapfree.py\\\': \\\'ba5588903711e311ba832840e9ca2e0aacff0ae778386e26eceea9459230be7f\\\', \\\'models/mickey/datasets/sampler.py\\\': \\\'1c367292c3ad74a7eb0235b22bf2844b07784286642d4e03e2e5a25b74631fef\\\', \\\'models/mickey/datasets/utils.py\\\': \\\'9338ea52e04053e1ca31226f394a51b33b884729adb1f39e0bdfc2162b7f4304\\\', \\\'models/mickey/models/MicKey/compute_pose.py\\\': \\\'046df083d87b009624df05a70daedab4d0b2cf18ef5f229365e8c22bdffc3822\\\', \\\'models/mickey/models/MicKey/model.py\\\': \\\'46f24ae476d63f4c9decff4f9e5979fc60a4bbff8124abd2379c9bcd2a04b014\\\', \\\'models/mickey/models/MicKey/modules/DINO_modules/dinov2.py\\\': \\\'69b25d9b374722f923c589ef7a44a2e552f7bf912fbde27aaf98e2cf63fb352c\\\', \\\'models/mickey/models/MicKey/modules/DINO_modules/layers/__init__.py\\\': \\\'34312c0841b7ab254915b34b197072ade6984e28e2c718315793bdbaa34339bc\\\', \\\'models/mickey/models/MicKey/modules/DINO_modules/layers/attention.py\\\': \\\'f7c67d57b0871fce8517dd50b55c02f33aa24c20493eff26971d3ec4cca0ec12\\\', \\\'models/mickey/models/MicKey/modules/DINO_modules/layers/block.py\\\': \\\'a7218ff8350794efce37a010ac7d58d212e10a5b84b43a214a7b682cac656a62\\\', \\\'models/mickey/models/MicKey/modules/DINO_modules/layers/dino_head.py\\\': \\\'5e1ca23565a0bcfd2ac1d79bfbe67c845e50582f4efe0bb1df2324bf4d4d98a8\\\', \\\'models/mickey/models/MicKey/modules/DINO_modules/layers/drop_path.py\\\': \\\'27abed4dd9dafa14157c9c7717bdca15fedb62702066f627b314bc91c06a1d53\\\', \\\'models/mickey/models/MicKey/modules/DINO_modules/layers/layer_scale.py\\\': \\\'a79f5b673c3f1fa60173b1905501ecfeb7fe77bed9b5e35881390c12baecbdfa\\\', \\\'models/mickey/models/MicKey/modules/DINO_modules/layers/mlp.py\\\': \\\'9856d7274ca32428ff05dacf7adf8ad10fa81d7bb4e5d299ab57cf421dcc7620\\\', \\\'models/mickey/models/MicKey/modules/DINO_modules/layers/patch_embed.py\\\': \\\'2d949ea0eff33185d829995373c963cac6eaebf77b4808478a72934d89587e42\\\', \\\'models/mickey/models/MicKey/modules/DINO_modules/layers/swiglu_ffn.py\\\': \\\'e1744aa9d29610c57d4fa153ec0aa545ef60da565d2aaac754d318fea0e30d7c\\\', \\\'models/mickey/models/MicKey/modules/att_layers/attention.py\\\': \\\'bb19faec56dc2eeb28e0535d4d3f1525ed0480dc881fefc22eff69335f5bfe25\\\', \\\'models/mickey/models/MicKey/modules/att_layers/transformer.py\\\': \\\'66e8fc630656d4c6182b6f7d95d05c33db76422faccf9e630166ee5a42cd841f\\\', \\\'models/mickey/models/MicKey/modules/att_layers/transformer_utils.py\\\': \\\'42c12024731c94f6d42d66322e08c1760ae297a963a2c5682073901059753123\\\', \\\'models/mickey/models/MicKey/modules/compute_correspondences.py\\\': \\\'d560f0ef016df950f2b954123cbcd46c14efc1be169e3f62365e5f2b09fcb244\\\', \\\'models/mickey/models/MicKey/modules/loss/loss_class.py\\\': \\\'1eb3f3b5a3dde9627df89eda045d5d874f68f21253309c76849999e219094da4\\\', \\\'models/mickey/models/MicKey/modules/loss/loss_utils.py\\\': \\\'44f3e1f84938aac4ffa5ca52ac740c7dd2de8db5514d12e83d305f8ca11fea42\\\', \\\'models/mickey/models/MicKey/modules/loss/solvers.py\\\': \\\'5e5a561c77834f694b4555b1c2dabf724723a9adee9c29766c347c16ce6ba308\\\', \\\'models/mickey/models/MicKey/modules/mickey_extractor.py\\\': \\\'7f9014c1c6179f16895c5a802da0174822a231fded0bb33a16d7eb98076764f6\\\', \\\'models/mickey/models/MicKey/modules/utils/extractor_utils.py\\\': \\\'0499531eaf76cb1a2ce985cc7b33ba734a240044133a3f15e5b53ad3dd705a7a\\\', \\\'models/mickey/models/MicKey/modules/utils/feature_matcher.py\\\': \\\'fe9c02506bfa3729cdc970be07e8bc153c2e7d9a1acf2664ee53eabe8c280592\\\', \\\'models/mickey/models/MicKey/modules/utils/probabilisticProcrustes.py\\\': \\\'06ab13b4ec87c0d443b4555f010b134f21fb9fd8245351b07f961b63426af771\\\', \\\'models/mickey/models/MicKey/modules/utils/training_utils.py\\\': \\\'455d37ce44761be09f6eee6d07b990b230ab04840be28af47468cfbf4c13e41a\\\', \\\'models/mickey/models/builder.py\\\': \\\'116d4a51ee43f62650b75237a06f239fc87d07011799112d5ee1d076387c51fa\\\', \\\'models/mickey/utils/data.py\\\': \\\'cbd63f3e34a4b6689755fe2cb3b52fabf51135a56bdf0c96b7f9c599d1b3a941\\\', \\\'models/mickey/utils/metrics.py\\\': \\\'7465d84b837115eb0e1c6f884492c0baa36a99e34a3b1073fa24c9f91e0131a1\\\', \\\'models/minima/depth_model.py\\\': \\\'c37dec1f9c84feab0c1899e7c847ecb99aab887d564d40087db913ce0e5060af\\\', \\\'models/minima/model.py\\\': \\\'a4a2445a7b7848d99adfd3a361ebbcb2aee533bfef2792280440c0e51716e8fd\\\', \\\'models/moge/feature.py\\\': \\\'b5a6a7e0ee30a43d5afb25b2f91d155d5ea73cc595900f6829d589c8aea92cfc\\\', \\\'models/mtldesc/__init__.py\\\': \\\'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855\\\', \\\'models/mtldesc/mtldesc.py\\\': \\\'30c829a32fa10b441f8c1aef44cd817161fc09ec5a22649953158018da07a631\\\', \\\'models/mtldesc/mtldesc_custom.py\\\': \\\'f677be6408969779b5c35fbe27a58f7eeb70233785b090658d3a94ce6798968d\\\', \\\'models/mtldesc/network.py\\\': \\\'4fda0a968aaeedd6b1a92bd23d0ee6677c80a866682a99079d510ba24390441e\\\', \\\'models/mtldesc/vit/__init__.py\\\': \\\'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855\\\', \\\'models/mtldesc/vit/vit_seg_configs.py\\\': \\\'f68d8cabec6f1eb75ea144f06472fb3692f802dc468ef6aa2501c092d09ee3c4\\\', \\\'models/mtldesc/vit/vit_seg_modeling.py\\\': \\\'ecbdfb141e9cfea9460b840cb00ea9d175cabcb3a6476a4d262485fa21716a86\\\', \\\'models/mtldesc/vit/vit_seg_modeling_resnet_skip.py\\\': \\\'8474e54f2d225edbf35f746b3346131df9501f172cc7d849818bbeb20961b949\\\', \\\'models/ncmnet/config.py\\\': \\\'3796235f91d553fdb4ad5436be8296f355542f1201f0f06fdbeba958cae9bfe8\\\', \\\'models/ncmnet/loss.py\\\': \\\'c2d71f05bf4b8794b9e0f25fd2946bbd71fded3f232f2c4a520121c4dbe11079\\\', \\\'models/ncmnet/main.py\\\': \\\'bf3a5ef024e2a14ba222e0838aedf4969cb3bf0153421c3c21f43663027a02ba\\\', \\\'models/ncmnet/ncmnet.py\\\': \\\'b62ac27b949001c7b9f0f8cd7fd9b96af1f09a204d19222f9eb230b187a8c633\\\', \\\'models/omniglue_onnx/__init__.py\\\': \\\'af6a5556205220b6ca3afacbc41450362f1b2abec398d5b759b381027a754e93\\\', \\\'models/omniglue_onnx/dino_extract.py\\\': \\\'5d142dff7965b64b6cc6bf07dcb9176ae44a7ac64bae1779e5361b37de86a6cd\\\', \\\'models/omniglue_onnx/dinov2/__init__.py\\\': \\\'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855\\\', \\\'models/omniglue_onnx/dinov2/dino.py\\\': \\\'849a1935fa14809e3028effc0b08821ff51a36e4895cba00a285013a85e098d8\\\', \\\'models/omniglue_onnx/dinov2/dino_utils.py\\\': \\\'ecb590178cd3e1d3e8c7c5744abff18a5898b37817bd7478f6a4f64e0327e7f8\\\', \\\'models/omniglue_onnx/omniglue_extract.py\\\': \\\'1b92b116e61971ce63ea47043de0d747a484449cd0c84425fd14bdc5e0ddbb88\\\', \\\'models/omniglue_onnx/superpoint_extract.py\\\': \\\'3b05434b3fa4098c246d7992b607214b8ad61ee2b420a13edbf49a714315e646\\\', \\\'models/omniglue_onnx/utils.py\\\': \\\'e85ae8799e65dc3d3e8288ed576073f02d00f6130bba12eb1f4ff1f3d7659786\\\', \\\'models/openglue/__init__.py\\\': \\\'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855\\\', \\\'models/openglue/inference.py\\\': \\\'e3e290b932fe04a9a4a31ef2a6a12bba2e397395af2edf9e81cfbaaeb845fd9b\\\', \\\'models/openglue/models/features/__init__.py\\\': \\\'66d1cb173ad989aa6152cd796acc0a6dab0d5006fba5aff9be523ce2c691643e\\\', \\\'models/openglue/models/features/aliked.py\\\': \\\'bc954bcf87a236cabf4f8506631a8340e339bd2a5087d2c4bf0cf47dc5bb0c82\\\', \\\'models/openglue/models/features/base.py\\\': \\\'d134554ba4b50bd341713160bd4e95785fc8de2156e35d2fb11deb12e6751d51\\\', \\\'models/openglue/models/features/hardnet.py\\\': \\\'a6c426d4835c2709b67b9b7f751601b6a91481f67400b76375b49ff631049ad5\\\', \\\'models/openglue/models/features/iterative_features_extractor.py\\\': \\\'f0a263be9b537dba3b735b75710abe5b20a6c96533fef1e6a2b84ec4f9ff1e15\\\', \\\'models/openglue/models/features/mtldesc.py\\\': \\\'df0a65c0846a62fa7ba44d7e67379a2d8a90765440c99b0ce2759a11d0722e81\\\', \\\'models/openglue/models/features/opencv/__init__.py\\\': \\\'7db2e780358707c33c7ebc8e27d5eea48c52929e34ae2a4cea3531d0ed542292\\\', \\\'models/openglue/models/features/opencv/_features.py\\\': \\\'1aee2b82225a7315b6dba6f3f6e459e2bc7f1b7563faced30a60590571f494d1\\\', \\\'models/openglue/models/features/opencv/_features_torch.py\\\': \\\'1814469910dc328d179322425e0ba99390ffd2f0558dfa9e57357c1255d2e3cf\\\', \\\'models/openglue/models/features/opencv/base.py\\\': \\\'18daa59731107b5fd852203a9a72d4163668abc8c295f5be7b24c8032eec9aec\\\', \\\'models/openglue/models/features/opencv/dog_affnet_harnet.py\\\': \\\'b93f3d4ff929d3a651a15ac29e1f9a4d455171a6f1a60ca6e28b0acd8f3291d5\\\', \\\'models/openglue/models/features/opencv/torch_wrapper.py\\\': \\\'bca32cd4902e56bf7c25305e3b7840644cf68f71f102564e7d75bbadf6333a69\\\', \\\'models/openglue/models/features/posfeat.py\\\': \\\'e912e934191a904242028af385e9ab15493c89cf3e60aa35997ec87148f90ee6\\\', \\\'models/openglue/models/features/sift.py\\\': \\\'496184201eba823d4ee218b58c9b7979efaf06bc3bf19677bd89d48b92aaca1d\\\', \\\'models/openglue/models/features/superpoint/__init__.py\\\': \\\'c241597f649fd2766052c5fd472ee7e4fce61fd23e807f0799a4dcaf1afe100d\\\', \\\'models/openglue/models/features/superpoint/model.py\\\': \\\'2dcc9d9117932c452802f408c452ae6efc5396baddc6ab543d72c4912382b74f\\\', \\\'models/openglue/models/features/superpoint/utils.py\\\': \\\'bf1f064dd58cc83678549d5b602ac949eacfd56291a4a4f807eb138827e3f03a\\\', \\\'models/openglue/models/features/utils.py\\\': \\\'d5c06a8ff52310a42baf5c742006a7fa8078192ca5e81babb17d96f139b3a6aa\\\', \\\'models/openglue/models/gt_matches_generation.py\\\': \\\'74ab32c07aa8da599456966d6be1a74929d9b59f4a80b2248599792e0fee0067\\\', \\\'models/openglue/models/laf_converter.py\\\': \\\'73b53cde47c8f85aaf623022a85079d5c49a2ffc29fe4dd4aafc768c0e3c4ac7\\\', \\\'models/openglue/models/superglue/__init__.py\\\': \\\'ac7a7eaf2ec06207aa8fdfbc8462836f7e6eaf43e51c831a8f9370f35fcedea7\\\', \\\'models/openglue/models/superglue/attention.py\\\': \\\'58a3d3fdf9e44fede529ef117906fe217f3f1da1448e5bf99b11196dead34b7b\\\', \\\'models/openglue/models/superglue/attention_gnn.py\\\': \\\'0f466e6d81888300d59e0284157c159c6b3a16e30f13725648f475f72530ddf5\\\', \\\'models/openglue/models/superglue/optimal_transport.py\\\': \\\'8500a2cd95fa667df8b54cf68cdb0d5d19295a7990b1965ff382d97497d9bb74\\\', \\\'models/openglue/models/superglue/positional_encoding.py\\\': \\\'84147035f2f18ddfb0b2f0ae5d46f010a1bdb2598b3980946cf12dcbe357692a\\\', \\\'models/openglue/models/superglue/superglue.py\\\': \\\'740cfce1115d290a8fa278dc648b5f2c272ee8012f3f913fea384df3988f0395\\\', \\\'models/openglue/models/utils.py\\\': \\\'0cca4b8126a3ed698aaedda3c7946eae79f2f0bea0b2d3968fb3d32a97403daf\\\', \\\'models/openglue/utils/__init__.py\\\': \\\'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855\\\', \\\'models/openglue/utils/augmentations.py\\\': \\\'18b944ce2ad5cfa4115d27180251c5c8d8ca8c64d84eccc86ef3fe2c71f5b544\\\', \\\'models/openglue/utils/lightning_callbacks.py\\\': \\\'b7f4f2ac1d7f0df22ed2a908a649b93570b1c6c16acd48a4b22988d8978b9fd0\\\', \\\'models/openglue/utils/losses.py\\\': \\\'43548ff23e1ac3b6610bcf7a9c0fb8e34ca84b648c4cc853cbb4b177c8abb6c6\\\', \\\'models/openglue/utils/metrics.py\\\': \\\'4b3d4a2f735f4a2e1b91b364ce998e95e6fd8c11ed73bce27b777e441e8462fc\\\', \\\'models/openglue/utils/misc.py\\\': \\\'11ae02cbb52cc680b8143d15d3e5383085b3b821ef13906edcb3deb06cedf971\\\', \\\'models/openglue/utils/train_utils.py\\\': \\\'087cdf5dc0a4d75e7b5fedd788dbff6b29730af5c017f4c4c3de18f9fdc89bb3\\\', \\\'models/panet/model.py\\\': \\\'ca80749e7652e5713f842d47cbb9e91b03759b3245080ec1a959a2a8ae6ef84d\\\', \\\'models/panet/refinement.py\\\': \\\'0ba687362ae799209f7626177d17fc09a036336e474dca2d2b4021deffaa0673\\\', \\\'models/patchnetvlad/model.py\\\': \\\'b46905838367605e555b48bc11c69591a65af0e3d654eb9009e9d009b562aaad\\\', \\\'models/patchnetvlad/models/__init__.py\\\': \\\'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855\\\', \\\'models/patchnetvlad/models/local_matcher.py\\\': \\\'9782134d5d595560a2dba5bba27485c9cbae9d1ced57791fe1ee154d3a2324e8\\\', \\\'models/patchnetvlad/models/models_generic.py\\\': \\\'99f61988e88949209213c8fc60eede106bb81110b5ad4ddca123444c82fef199\\\', \\\'models/patchnetvlad/models/netvlad.py\\\': \\\'181373dac1835665b2e8be2e19150356bf9e922cc1e43e1e03b753421bffc75c\\\', \\\'models/patchnetvlad/models/patchnetvlad.py\\\': \\\'0565cea2eb8012177d82b5f1523b09281de321f78c47f377d5f455da4298c6ad\\\', \\\'models/patchnetvlad/tools/__init__.py\\\': \\\'774ba53759632b8be54f28b8f54ed5a9584d5e7daccf17475ebf0b15e946c458\\\', \\\'models/patchnetvlad/tools/datasets.py\\\': \\\'eeef2ef48c8409d3a1c7378199ea9c65737fcd66741f29d3b23a8638f47cde06\\\', \\\'models/patchnetvlad/tools/gen_image_file_list.py\\\': \\\'7d11f99da17ab4a64599fec50a008e92617c3a536ba00fad26c764ba82b20a55\\\', \\\'models/patchnetvlad/tools/patch_matcher.py\\\': \\\'bebdf031162a51511b6f5e5e9e67e1370129b5b94594ab713feb395a75104404\\\', \\\'models/pixloc/__init__.py\\\': \\\'1c9a8ff48eae4e742df4ca3c94c037970f4d5ab6769bc8f7d08d8d969067a79f\\\', \\\'models/pixloc/localization/__init__.py\\\': \\\'b791e0b44ecb6c666dc5a1eb8fb5edb05f1d99b40e0f31fb9d285323056eb7f4\\\', \\\'models/pixloc/localization/base_refiner.py\\\': \\\'1467d6b69d95fb7cd5eb03edab440a54681b2c8059811294096ce18783b90c86\\\', \\\'models/pixloc/localization/feature_extractor.py\\\': \\\'b57f8b66518ab72bd7c483e9c3b77bf7e78d24fcdce3824225b6e122f323bca7\\\', \\\'models/pixloc/localization/localizer.py\\\': \\\'9a15a4b6e8ce128ccb2ece5db59b354eaab25cf3d580982b61a8cde7c64ed99f\\\', \\\'models/pixloc/localization/localizer2.py\\\': \\\'02bc105c216572916a854dda9d776a259ea2927011690176d229667dbf447c23\\\', \\\'models/pixloc/localization/model3d.py\\\': \\\'4477b2efe60d8a70966d3cd028ef4f702158cd01fc9f841ab53fbc8e0a2c6034\\\', \\\'models/pixloc/localization/refiners.py\\\': \\\'735225ea19d2cdbf71d945d2c171bafe2b32cce48c3181d25408df76efc204a8\\\', \\\'models/pixloc/localization/tracker.py\\\': \\\'71a0a0ddd5ab650786d39a1f3bc077215d8d26a0eb6ae1c9f9f33dc8194a9cdc\\\', \\\'models/pixloc/pixlib/__init__.py\\\': \\\'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855\\\', \\\'models/pixloc/pixlib/datasets/__init__.py\\\': \\\'5afe2856e7644170c67ae811c63309e2e69eacb63d3307a01c8cb13a87dc6ab7\\\', \\\'models/pixloc/pixlib/datasets/base_dataset.py\\\': \\\'71cfc09f6a934160b553761def721746cdc09557d6ba93c1cf3a93bd47a66875\\\', \\\'models/pixloc/pixlib/datasets/cmu.py\\\': \\\'20dd7c6c6fbfa671bde035583c2d8de6c8259f0723b54c299bd10377a7e6bc44\\\', \\\'models/pixloc/pixlib/datasets/image_folder.py\\\': \\\'7166d89130c1a1714c9b43d80aad9d79640d8a133a50df6b17b46933fe87b647\\\', \\\'models/pixloc/pixlib/datasets/megadepth.py\\\': \\\'f391971c26fd1ab3c9af16ef233fcaadbcb7029379ba5c04ae8703ba4ac25544\\\', \\\'models/pixloc/pixlib/datasets/sampling.py\\\': \\\'fd7ac60da1306ef92b629dde98750fa21fc279d80fe296b4ff92a81289257032\\\', \\\'models/pixloc/pixlib/datasets/view.py\\\': \\\'38cd75f4fcbfdefe7700b886736ae9e3fcb6c037fe6f2bff86db81fdfdaac83a\\\', \\\'models/pixloc/pixlib/geometry/__init__.py\\\': \\\'6bf2dc9d94081863e24259c2a0a5c92072ed048ddeaf7691a84b217d96b0e32d\\\', \\\'models/pixloc/pixlib/geometry/check_jacobians.py\\\': \\\'e491690c8393ca58bf97598e8201d577c68ec56f8f65e4d2e6fe215177ab1490\\\', \\\'models/pixloc/pixlib/geometry/costs.py\\\': \\\'5426362bb5e71016fe46ae272f214726d7195202a04ccef9673bf4d56233b6ca\\\', \\\'models/pixloc/pixlib/geometry/interpolation.py\\\': \\\'317d6bed3d7479758f563c69e0c1ed11a556ca58da313f90440c80e0b792cf55\\\', \\\'models/pixloc/pixlib/geometry/losses.py\\\': \\\'10b34b1a62c76ce60cc0b3ab38eadba06b63aee0f712fdc663fad96ea82e4f4d\\\', \\\'models/pixloc/pixlib/geometry/optimization.py\\\': \\\'f1bd3c5dbbb46a712d8d88b0dbbd906f2a1810ac9c35714ef4fa29bf38f5f6d0\\\', \\\'models/pixloc/pixlib/geometry/utils.py\\\': \\\'20a7b91bc61f6844c65b98c687f89b74c1660d7bdbc63c57dc5c7f8d2197f73e\\\', \\\'models/pixloc/pixlib/geometry/wrappers.py\\\': \\\'696b36efc21a78dc00e005895ba45a4578cf9886a3c37e7bada9e6d40b8605da\\\', \\\'models/pixloc/pixlib/models/__init__.py\\\': \\\'12112096cd5588d20c68bfa2473b562ae81eed142d8d1d1d7c3d9abe55193aaa\\\', \\\'models/pixloc/pixlib/models/base_model.py\\\': \\\'8aabc315182078b78cb4d5aeea348b742b473702864a08428f65d241e00df8d5\\\', \\\'models/pixloc/pixlib/models/base_optimizer.py\\\': \\\'4764e625a10fbb6d92ff62aecbf7e00f838c0f7a938e1c4a5ee39495888e4f5c\\\', \\\'models/pixloc/pixlib/models/classic_optimizer.py\\\': \\\'747196033fc827de2bd5bbb05c34e9c11c3284a709a169d072f2394c978a57d6\\\', \\\'models/pixloc/pixlib/models/gaussiannet.py\\\': \\\'af09970a32255acb47f22b2c55be7f48c4c526532efd1652bce8ed00590ed317\\\', \\\'models/pixloc/pixlib/models/gnnet.py\\\': \\\'734c229603de3b4540ae48cd6fd69e8a4c1242d4e2a3d506bbabf6550ddbca2a\\\', \\\'models/pixloc/pixlib/models/learned_optimizer.py\\\': \\\'eaba5adc836ad705f0b816bcc37fe9ff6b6dbd1fd61951bffc9cadc0050bffaa\\\', \\\'models/pixloc/pixlib/models/s2dnet.py\\\': \\\'a3da910544a26bb5b7b6dda87e14fb39c0fcea61de6e6c629ced4c194442fab3\\\', \\\'models/pixloc/pixlib/models/two_view_refiner.py\\\': \\\'e8b1b3957fdc778e3b6b6fbc127d0bdc0076f83762048d532a3212c2875b5934\\\', \\\'models/pixloc/pixlib/models/unet.py\\\': \\\'784a0903fb1b9d36e9fcc096d92f9b7ba2be768b8e302a4fb2c4ac0d798902aa\\\', \\\'models/pixloc/pixlib/models/utils.py\\\': \\\'f080f61ba170dc39bd3e62b0a2b17f0a16c4da54aa6d4317a7698e2127358fca\\\', \\\'models/pixloc/pixlib/preprocess_cmu.py\\\': \\\'93b7db986485f479c711096a83d8b21fea212eb7cf3c9c341f92912604634035\\\', \\\'models/pixloc/pixlib/preprocess_megadepth.py\\\': \\\'72438616efddb6c0bc4bf220b25fd97eafb0df39c01047d30c964f6adc2d5310\\\', \\\'models/pixloc/pixlib/train.py\\\': \\\'b7d10d840d777086f96a776cf04cdd9da5e874b9578f2545065a219069515a7e\\\', \\\'models/pixloc/pixlib/utils/__init__.py\\\': \\\'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855\\\', \\\'models/pixloc/pixlib/utils/experiments.py\\\': \\\'31d510484fd87be744f88e28316642ac46ad212471e18e0a4d670828b1167b51\\\', \\\'models/pixloc/pixlib/utils/stdout_capturing.py\\\': \\\'4160daf5896808d16b98fb3f913c00214e0aa9a67de340653a0967b041d30c35\\\', \\\'models/pixloc/pixlib/utils/tensor.py\\\': \\\'c9d63cfc244537e2c50ca93692b5651f54525720fbbb711731568f76b6f41210\\\', \\\'models/pixloc/pixlib/utils/tools.py\\\': \\\'9bc19d1a981b6e8006f200527d38e46be07494902defe99c032baef166f8affa\\\', \\\'models/pixloc/settings.py\\\': \\\'a6a598cc4b658208cd1f9c0f6dec82ef4a3dfd5cc9f150bb14b60fca21f1f3c3\\\', \\\'models/pixloc/utils/colmap.py\\\': \\\'46d65227f0decff703cf59b7f94b845292ffd6ad54ac4329a4d8d43d70faa5df\\\', \\\'models/pixloc/utils/data.py\\\': \\\'938453f0c08efc8b31eb00cebef978dd6ad0b00e715a92668ea5aeb3e4f52118\\\', \\\'models/pixloc/utils/eval.py\\\': \\\'f71fe82d557abe467202294adbb67526c3d4135f376ac2a1f2731508679dce0c\\\', \\\'models/pixloc/utils/io.py\\\': \\\'8d38f60e108085ecd7abcf3c27d4fa8e9303be238d10a9ec6071eb1bf3a8f25c\\\', \\\'models/pixloc/utils/quaternions.py\\\': \\\'ff90c47fe9d35e4266370bbdd639a24c009844eec7eaddb43a3e2361dd207460\\\', \\\'models/pixloc/utils/tools.py\\\': \\\'a364b6c7f6316859a2cc7e5c87d90e37e69faba6b35c8d5ab0cadf31dba83417\\\', \\\'models/pixloc/visualization/animation.py\\\': \\\'b811a1fef82e2f83f7b2a5cd4cf240156a0f3f76ef3d70dcead8e6e507dab5bc\\\', \\\'models/pixloc/visualization/viz_2d.py\\\': \\\'a78ab6e0d1df41d9d6e74ef993408e2c1708f4508e4bce312623e510dca9f716\\\', \\\'models/pixloc/visualization/viz_3d.py\\\': \\\'a31052f0c711f8374e3a25dc6193335b99a02572fce3051139ec91d00c2d680c\\\', \\\'models/posfeat/extractor.py\\\': \\\'8e7693e04204628454f02d67ded6ddfd1bbbc880080a8b8f373802000faef9f9\\\', \\\'models/posfeat/losses/__init__.py\\\': \\\'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855\\\', \\\'models/posfeat/losses/epipolarloss.py\\\': \\\'5b2f703579ef33fd2a532b4fc3bd2a1a9ca78946a031ae32b9f1697403733c0d\\\', \\\'models/posfeat/losses/kploss.py\\\': \\\'4c7b98237c48c137046eb64bf66caef93dcdb7dae8cea74850686a3519979c93\\\', \\\'models/posfeat/losses/preprocess.py\\\': \\\'df90713138db72def24e1486354598bfe9c6c6d7e98c653fed24bfe8f1ead757\\\', \\\'models/posfeat/losses/preprocess_utils.py\\\': \\\'63ce80ae13bbd2c6906090080dfc298f81cabc693181b3cf295e330af0a90971\\\', \\\'models/posfeat/networks/DescNet.py\\\': \\\'e382cc730ed57d018697840fd78257dcbea8306a95d7070cd2289172997f66ea\\\', \\\'models/posfeat/networks/DeteNet.py\\\': \\\'abbeebcac726120cd52560383265f80086b7187f4e60a8ac29f438d637e7f627\\\', \\\'models/posfeat/networks/PoSFeat_model.py\\\': \\\'84e4a309d4751bac23710678c19aa1fe253a6c05eb79b2e7d116e018d48d0b20\\\', \\\'models/posfeat/networks/__init__.py\\\': \\\'e084ba77ddf3c9b34f4d47fac01a1ebf2f5a2cca39b15ebccd0a24ac53fa8e84\\\', \\\'models/quadtree/__init__.py\\\': \\\'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855\\\', \\\'models/quadtree/config/__init__.py\\\': \\\'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855\\\', \\\'models/quadtree/config/default.py\\\': \\\'44922e3f10c757ee968b1c0261ed649b7ffa14e4feb031de8e2c0af14121df36\\\', \\\'models/quadtree/functions/quadtree_attention.py\\\': \\\'a33c558ccabdbac8db803c4945408e520a15d86936cab20f2c14baf5ba51cfd0\\\', \\\'models/quadtree/lightning/__init__.py\\\': \\\'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855\\\', \\\'models/quadtree/lightning/lightning_loftr.py\\\': \\\'c54e74f1ee2a48e54d0b1d9f25eb8d84aba9818772d04510a792421df09f93ee\\\', \\\'models/quadtree/loftr/__init__.py\\\': \\\'7ae3e08968ed7981c5fdc6a78f1d71d6b89f55f33c42b68b90667aeb60bc2d50\\\', \\\'models/quadtree/loftr/backbone/__init__.py\\\': \\\'6e34cdcc99f4f6bafa5a48d934d3a2052b739d913a9d3c37e6921344c76abe56\\\', \\\'models/quadtree/loftr/backbone/resnet_fpn.py\\\': \\\'997d9778915524defd9f7e36b31992c811b85b215a13291af5ec781dc1a86061\\\', \\\'models/quadtree/loftr/loftr.py\\\': \\\'22047d2b1dad885c3764397060c15f49a75bb3ab3e782b0f651ff9a1dc947172\\\', \\\'models/quadtree/loftr/loftr_module/__init__.py\\\': \\\'25fbc03b34ce660b9534db639a7a9b21f30c9450922671a91bb226ede6bce3b0\\\', \\\'models/quadtree/loftr/loftr_module/fine_preprocess.py\\\': \\\'1eeb83778a4c50b5752ff37d2b8ac72d76706ea4fe3bc222ab8c47263279fafe\\\', \\\'models/quadtree/loftr/loftr_module/linear_attention.py\\\': \\\'0bb92eeb20133ae75ad489c27f563a8a0973afeef48a5072098845684af04ef1\\\', \\\'models/quadtree/loftr/loftr_module/quadtree_attention.py\\\': \\\'574f38da1e158e241280ea3d790d83b3500a3fa632093e3ba207c26d428860a2\\\', \\\'models/quadtree/loftr/loftr_module/transformer.py\\\': \\\'3da9be56bb21926da298e392a1fdab47f7ccb7be21e066b7a3755c5d1f80c6a0\\\', \\\'models/quadtree/loftr/utils/__init__.py\\\': \\\'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855\\\', \\\'models/quadtree/loftr/utils/coarse_matching.py\\\': \\\'df1c7cbff72de0c1a521f6a0fbb2fd2f0e167b9677a8ed366970b457e56a4adf\\\', \\\'models/quadtree/loftr/utils/cvpr_ds_config.py\\\': \\\'06b0578182cb58431fad6e9bfb2ae5680e777851add70f0497e2c660e45d00f4\\\', \\\'models/quadtree/loftr/utils/fine_matching.py\\\': \\\'823dd17053a8559f7b1ca32e41b46552c587c295ae1ab988ad75d2c91448cf3a\\\', \\\'models/quadtree/loftr/utils/geometry.py\\\': \\\'d10afe85e7144777343ff193d16baab514683a19f207832422f15bc771226668\\\', \\\'models/quadtree/loftr/utils/position_encoding.py\\\': \\\'285f4a55ee46d2ec01166250b0a6ed36d7e584aa48094311734b6f2933979d91\\\', \\\'models/quadtree/loftr/utils/supervision.py\\\': \\\'d646d1579b2dd345f0c0cfc8c607b1c156810e5ad0fd058f8b0bd45780cce420\\\', \\\'models/quadtree/losses/__init__.py\\\': \\\'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855\\\', \\\'models/quadtree/losses/loftr_loss.py\\\': \\\'50523dd2e141cd3f967afd84f4888afcebe6158c6203530807428d6858433257\\\', \\\'models/quadtree/modules/__init__.py\\\': \\\'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855\\\', \\\'models/quadtree/modules/quadtree_attention.py\\\': \\\'5072e4a66ba0e522bd315cf16dc9ccedf5a5d2234f83aeb0b30f879b644baaa8\\\', \\\'models/quadtree/optimizers/__init__.py\\\': \\\'de9fd78fedfdb1d7071ea56c512ecb5ac5bb8760b0cd781e5c250646f1d57a46\\\', \\\'models/quadtree/utils/__init__.py\\\': \\\'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855\\\', \\\'models/quadtree/utils/misc.py\\\': \\\'a930e98547ea24852d571bdeb7d9c22b342d0ccf3823e02a91a92169b69b3e64\\\', \\\'models/quadtree/utils/profiler.py\\\': \\\'f900e65ac91eae78cbf2d57f1b0469dba2b9727e066fcd7c92dbce61d4b32cc5\\\', \\\'models/relf/descriptor_utils.py\\\': \\\'dcc41ae377c4aecd0890e755fc126b9e650d45067b3b381c8c21f273323a255a\\\', \\\'models/relf/group_align.py\\\': \\\'48f8777b79bb30912d5b65f2a5432ec1d518b08394960b0e9213b66982faa5fd\\\', \\\'models/relf/load_model.py\\\': \\\'700ae501ccf60d34795c1d0aebf236925fffea804852c6db5bec0ba2733b3ff6\\\', \\\'models/relf/loss.py\\\': \\\'dbf98774e10f8438ebf18533439b8d5710784e8a511e9182cf7102b9d58606bd\\\', \\\'models/relf/model.py\\\': \\\'502fa00cf64464893c232cbc231098272e528ba7bae5a121f861bc4a5fdfaa9b\\\', \\\'models/relf/reresnet_model/model.py\\\': \\\'1f839deefb3ee7da42eb2347e3f2a2e1d5279040872992ac4f9458789b8b4bc7\\\', \\\'models/relf/reresnet_model/models/backbones/base_backbone.py\\\': \\\'385a67f15d2a3e3ef17e1b134879a476959c707ac08f3cf101921eeeff444fce\\\', \\\'models/relf/reresnet_model/models/backbones/re_resnet.py\\\': \\\'d7bf8c88fa5b39f5002f452f325e07d0bd19da94268736f28e365e81630ad47f\\\', \\\'models/relf/reresnet_model/models/builder.py\\\': \\\'097c99364207516e5b875caf1b635481f7452f1c8a83c11c60752df21320656f\\\', \\\'models/relf/reresnet_model/models/registry.py\\\': \\\'659df646dae20f98929fe5bbd67d08c29dc5cc098f25c2d7c90e72ed9e5a0bd8\\\', \\\'models/relf/rewrn_model/model.py\\\': \\\'823e253cc8db55d68e55aa34275716b25cc976e0735a7fed85b3a695f23b5e36\\\', \\\'models/relf/rewrn_model/models/__init__.py\\\': \\\'be07616cf384f08cd3de313e82597a767e40a4e10702208251590e80ed897ebc\\\', \\\'models/relf/rewrn_model/models/e2_wide_resnet.py\\\': \\\'0be5862ed2a3cc7f35479f0e742da540ddecef019e5d9bad1a853f2c5dd49f18\\\', \\\'models/relf/rewrn_model/models/e2sfcnn.py\\\': \\\'49bccf27af1bfbed3938ae1ac985ba5c8ff5ab25c7c9f9b5556f71da91943b8b\\\', \\\'models/relf/rewrn_model/models/e2sfcnn_quotient.py\\\': \\\'847e26e0c229e53f3a4656535df0e324598ebf3f8e7f9b1108883e9d8311fccb\\\', \\\'models/relf/rewrn_model/models/example.py\\\': \\\'9f77fe0524329dc49d064b103e38a538cc8ea3622c2cab116412e6cbc081c8c2\\\', \\\'models/relf/rewrn_model/models/exp_cnn.py\\\': \\\'cfb20415970ad3212e11f082385e1abb59f620bf711ab16e621a90176ffd0d85\\\', \\\'models/relf/rewrn_model/models/exp_e2sfcnn.py\\\': \\\'c1c3c29048958292c43cbf7873e563293911735f19332a9e585175918c18b5a9\\\', \\\'models/relf/rewrn_model/models/utils.py\\\': \\\'d28bdd7a372140a3977759de1315ae86a1bf5270e66ab3d93849171d0dc6d852\\\', \\\'models/relf/rewrn_model/models/wide_resnet.py\\\': \\\'f5b7f14cdc2ecd063adc7bc41319d32f7203680f3239bfd34cbee544c40546e5\\\', \\\'models/relf/utils/evaluate_utils.py\\\': \\\'2804f8c49351ebefeeec37458c25509693584d2819281c7597670389ab466fae\\\', \\\'models/relf/utils/extract_utils.py\\\': \\\'9066a6ce45e2ec2546c106fba5b238fa55bf861bf026a4bd4722f19ad418b9f0\\\', \\\'models/relf/utils/logger.py\\\': \\\'40891d6e1a9c71bc55f8000501a154565f92b105d9b3ed6fe96036a20b543814\\\', \\\'models/roma/models/__init__.py\\\': \\\'3e4899d824341d67fbae2f33b595aec8f663f6c180cbeb286ec5f89119e6bde3\\\', \\\'models/roma/models/encoders.py\\\': \\\'61266169d5b5395dfa02e87515dd027bfd533fdd267c8e8b69125b12a91a2cac\\\', \\\'models/roma/models/matcher.py\\\': \\\'50d8f88097a99b0876a34664f91947217b81e517ffa64da3e138495e2761e87c\\\', \\\'models/roma/models/model_zoo/__init__.py\\\': \\\'79a6ec7b3743ec5f0465e9247c9cc34ebd8b507dbdde2e42792330c7f46cd4d5\\\', \\\'models/roma/models/model_zoo/roma_models.py\\\': \\\'5e908f0b1748ee9959de803c44a27bc7cfa9b42f85c4d31b21a1b4994a33c92d\\\', \\\'models/roma/models/transformer/__init__.py\\\': \\\'96f19df9e7b3339e487b4a7f3fb0af8c17eeadf3c2ddd0bca9976e2e35f91d7b\\\', \\\'models/roma/models/transformer/dinov2.py\\\': \\\'4b7d20d2fc7dc725fbe081ca14ea3763cf02433bb3c2bcebb766b977b0e5edce\\\', \\\'models/roma/models/transformer/layers/__init__.py\\\': \\\'34312c0841b7ab254915b34b197072ade6984e28e2c718315793bdbaa34339bc\\\', \\\'models/roma/models/transformer/layers/attention.py\\\': \\\'f7c67d57b0871fce8517dd50b55c02f33aa24c20493eff26971d3ec4cca0ec12\\\', \\\'models/roma/models/transformer/layers/block.py\\\': \\\'a7218ff8350794efce37a010ac7d58d212e10a5b84b43a214a7b682cac656a62\\\', \\\'models/roma/models/transformer/layers/dino_head.py\\\': \\\'5e1ca23565a0bcfd2ac1d79bfbe67c845e50582f4efe0bb1df2324bf4d4d98a8\\\', \\\'models/roma/models/transformer/layers/drop_path.py\\\': \\\'27abed4dd9dafa14157c9c7717bdca15fedb62702066f627b314bc91c06a1d53\\\', \\\'models/roma/models/transformer/layers/layer_scale.py\\\': \\\'a79f5b673c3f1fa60173b1905501ecfeb7fe77bed9b5e35881390c12baecbdfa\\\', \\\'models/roma/models/transformer/layers/mlp.py\\\': \\\'9856d7274ca32428ff05dacf7adf8ad10fa81d7bb4e5d299ab57cf421dcc7620\\\', \\\'models/roma/models/transformer/layers/patch_embed.py\\\': \\\'2d949ea0eff33185d829995373c963cac6eaebf77b4808478a72934d89587e42\\\', \\\'models/roma/models/transformer/layers/swiglu_ffn.py\\\': \\\'e1744aa9d29610c57d4fa153ec0aa545ef60da565d2aaac754d318fea0e30d7c\\\', \\\'models/roma/utils/__init__.py\\\': \\\'0099c7261125df7a0b59798647806a92d09fcdb8e06470764f58332291dc3a27\\\', \\\'models/roma/utils/kde.py\\\': \\\'c3e665b9e09f5d64c6f4a2c699f77f54478ced0924e6f02f73b3e747c3b6ec22\\\', \\\'models/roma/utils/local_correlation.py\\\': \\\'a736b4fb133176ac737e89b9e3b8e647ab3d43b65abb5221051513f2034c020d\\\', \\\'models/roma/utils/transforms.py\\\': \\\'d87ed4f7877def17502720b81a7f585c2e3ea7f14781664716230479e189209e\\\', \\\'models/roma/utils/utils.py\\\': \\\'44f655cc9f27d9cb20169f85258f458dc932ded14c5cb432e543d6502ba4c50e\\\', \\\'models/rotation_steerers/matchers/dual_softmax_matcher.py\\\': \\\'4b9da399af31a7f53ca85a168625b2c3698bb595f480ba2f3ebf41262ec2fa2a\\\', \\\'models/rotation_steerers/matchers/max_matches.py\\\': \\\'39cadfa577bafff8c33f441bb8ed36fe53b7add8b87f9b48f6829a34c644369d\\\', \\\'models/rotation_steerers/matchers/max_similarity.py\\\': \\\'28942e763c4aca6e9ba4df3cc135378afa29eda5ffbe90154e94dcc0fecbbda2\\\', \\\'models/rotation_steerers/steerers.py\\\': \\\'72c765508a2aa4da718aa0a6961f0f21bad9f84c25572f44c55e808d8c7e4e99\\\', \\\'models/se2loftr/__init__.py\\\': \\\'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855\\\', \\\'models/se2loftr/config/default.py\\\': \\\'262c175b781f7f635b391fd6b408a6b07c34b23276038a25d8e9cb5afd6b6823\\\', \\\'models/se2loftr/loftr/__init__.py\\\': \\\'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855\\\', \\\'models/se2loftr/loftr/backbone/__init__.py\\\': \\\'b471141ac3179cbecf24bc2b1f24d6a6e89d3514a64b4381c0cdc54216f44634\\\', \\\'models/se2loftr/loftr/backbone/resnet_e2.py\\\': \\\'b5e92d6bac187398b510bf654abc158f41865d9575cb42a8b95c60b48388a54c\\\', \\\'models/se2loftr/loftr/backbone/resnet_fpn.py\\\': \\\'997d9778915524defd9f7e36b31992c811b85b215a13291af5ec781dc1a86061\\\', \\\'models/se2loftr/loftr/loftr.py\\\': \\\'4b4a86b28f23e3fe563b2d7abce7fa833c35d667ba1528a01b40bf88de6c3f31\\\', \\\'models/se2loftr/loftr/loftr_module/__init__.py\\\': \\\'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855\\\', \\\'models/se2loftr/loftr/loftr_module/fine_preprocess.py\\\': \\\'1eeb83778a4c50b5752ff37d2b8ac72d76706ea4fe3bc222ab8c47263279fafe\\\', \\\'models/se2loftr/loftr/loftr_module/linear_attention.py\\\': \\\'0bb92eeb20133ae75ad489c27f563a8a0973afeef48a5072098845684af04ef1\\\', \\\'models/se2loftr/loftr/loftr_module/transformer.py\\\': \\\'457dd86c2b667419b22c1908b1dd0b9f22fea6f1b7e9cdd2e3b518db341299ae\\\', \\\'models/se2loftr/loftr/utils/coarse_matching.py\\\': \\\'df1c7cbff72de0c1a521f6a0fbb2fd2f0e167b9677a8ed366970b457e56a4adf\\\', \\\'models/se2loftr/loftr/utils/cvpr_ds_config.py\\\': \\\'0fbf9ac264deb4c923e56e1e564246649ab6cdf2562a95815fe58083d8c05f22\\\', \\\'models/se2loftr/loftr/utils/fine_matching.py\\\': \\\'823dd17053a8559f7b1ca32e41b46552c587c295ae1ab988ad75d2c91448cf3a\\\', \\\'models/se2loftr/loftr/utils/geometry.py\\\': \\\'d10afe85e7144777343ff193d16baab514683a19f207832422f15bc771226668\\\', \\\'models/se2loftr/loftr/utils/position_encoding.py\\\': \\\'285f4a55ee46d2ec01166250b0a6ed36d7e584aa48094311734b6f2933979d91\\\', \\\'models/se2loftr/loftr/utils/supervision.py\\\': \\\'32172c7e44c4aa56f452aff8c97eebb4c44df0af5be25f4fa5aefe3a0bdb19ad\\\', \\\'models/sfd2/extractor.py\\\': \\\'61a6c4fed85071f9bfab406cd8ef53c8d253c7e6c721e27ee62c5876c97b90f8\\\', \\\'models/sfd2/sfd2.py\\\': \\\'564ab80af2d54c6394d8b62a1678b817f424149359c47f1fd5bd95032edb8ba9\\\', \\\'models/superpoint/__init__.py\\\': \\\'c241597f649fd2766052c5fd472ee7e4fce61fd23e807f0799a4dcaf1afe100d\\\', \\\'models/superpoint/model.py\\\': \\\'f257cd7c3e2f8c45df096a07a1d5e28a923ed2d0505728773dc3772b85c9d667\\\', \\\'models/superpoint/utils.py\\\': \\\'bf1f064dd58cc83678549d5b602ac949eacfd56291a4a4f807eb138827e3f03a\\\', \\\'models/vggt/custom_model.py\\\': \\\'2861378247131074fc11ee865cec0e7120da6d4dd12a32e1b5488e5daac9d335\\\', \\\'models/vggt/process.py\\\': \\\'05b8207f556b229a69dfa1242faaf114e26e8be6536a7117a2e58adb525a61c5\\\', \\\'models/xfeat/__init__.py\\\': \\\'4b7db279af357c2af4cbcc67d731d966c3520e27f364524bfa914f945b6dffc0\\\', \\\'models/xfeat/interpolator.py\\\': \\\'21011341948ac5a795a6fc0318b99b3552501e659b923875f38f2e618bc3d011\\\', \\\'models/xfeat/model.py\\\': \\\'c8120da5d8bc57650bd7bc45e8e6e5aa77d87650c416378b51a23035fa4aec63\\\', \\\'models/xfeat/xfeat.py\\\': \\\'2131e94afced029ed3eadd2debec28073dcb86fdc00c682d544a5649b0a1ebfe\\\', \\\'pipeline.py\\\': \\\'657267486686e919002aba6df78a5820c3f5179238ee9f6f361b67d75c1df9c8\\\', \\\'pipelines/__init__.py\\\': \\\'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855\\\', \\\'pipelines/base.py\\\': \\\'97c406388a42df20d6093de42f8afb714a233911db6f5d04a1bcd6abcac7e825\\\', \\\'pipelines/common.py\\\': \\\'b07129dbccb4fc4aaa9d155a25d446003c830e5f8f9895e56219e28de3eeeb9f\\\', \\\'pipelines/config.py\\\': \\\'d8799bc4488f154fafb1618927baee60a853dfedd3fe31cbf82090968cb9e9c8\\\', \\\'pipelines/imc2023/ann_ensemble_pipeline.py\\\': \\\'45e7a7513e89c17e939509bac91d00d1b4d0effc2164400fb105bfcd023cd6a8\\\', \\\'pipelines/imc2023/detector_based_pipeline.py\\\': \\\'bc79ac5932f3cd89f82796112a962d6bd3a71f3d620497a4279b99580b20c0e9\\\', \\\'pipelines/imc2023/detector_free_pipeline.py\\\': \\\'5cb6c131983a3b9c30813fe055bf46446e34c7121ebc11c4f3da68c7d6acb055\\\', \\\'pipelines/imc2023/prematching_ensemble_pipeline.py\\\': \\\'74ceade84f9f1822ed0a7a90b101aa1438c7f3b7404fe673ebb3753c0bf36e7a\\\', \\\'pipelines/imc2023/simple_ensemble_pipeline.py\\\': \\\'c54946c6d8ef1a8ecd69133e7c5c8afbea08364bd2eae1f278e678749e9ddfeb\\\', \\\'pipelines/imc2024/dust3r_pipeline.py\\\': \\\'050914a482651e10e3477f97959afc698f39d9d3ef496bd1ef69fcf03a025ab6\\\', \\\'pipelines/imc2024/imc2024_pipeline.py\\\': \\\'626bfc1244f2235aeee42bd3055978c168159e45eb476305b15b91028c7003d4\\\', \\\'pipelines/imc2024/kernel_debug_pipeline.py\\\': \\\'19c17670b11d14ab01e74923fbfe9dbf31c92ffc7dab859a0e4b31093951be21\\\', \\\'pipelines/imc2024/localizer_pipeline.py\\\': \\\'96dd3b5e59452241d02deb19deea6e06ad313b90ae02177315e433d9288388d0\\\', \\\'pipelines/imc2025_mast3r_pipeline.py\\\': \\\'71bd5d65b020c32017757cbeb52e5a91d7f7d5129d716d576b2ab6a8d253c771\\\', \\\'pipelines/imc2025_pipeline.py\\\': \\\'75298743010eb0f7ea46d72a6ce34ff17978da0dbec573b7ca3939f76d64d8be\\\', \\\'pipelines/matching.py\\\': \\\'5b20adee2762066dfda221c40da39cc8c59fc5c9f6d2faf331e6f9519e5f3cf9\\\', \\\'pipelines/scene.py\\\': \\\'9248324c70eee7984c17932fc0c8d502e571d7616d0437990dc51bcdf369eaf7\\\', \\\'pipelines/snapshot.py\\\': \\\'fae8224020118e6655891342284f84893c2d722e800d13c4b78b5fae30bf4f74\\\', \\\'pipelines/stage.py\\\': \\\'2f6c39d35e1d6172bb2874f1e8bf6a65302a172688d65c0cdc789b3b3d067b6e\\\', \\\'pipelines/verification.py\\\': \\\'d87652d31f3b04ba161c2a8136b73ce01f2f5a6785dcc9b2c1ca179053ac6d99\\\', \\\'pipelines/visualizer.py\\\': \\\'ab0cc4e99a9ab08b7645474fe1df8f7dbda6457594e541a6dbf5bc35f219870c\\\', \\\'postprocesses/__init__.py\\\': \\\'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855\\\', \\\'postprocesses/base.py\\\': \\\'be2e5867a6d46d2ee65c9f2e1f0ff38f1217aecb7c9ba33ac7202aef374bbbdb\\\', \\\'postprocesses/coarse_match.py\\\': \\\'41f3a2d602ec09256b108fd6bd87c1ba64b76d4233d0b20fd9dff66a7764f1c9\\\', \\\'postprocesses/config.py\\\': \\\'6e2a2643067822e3108391e182e00a055d41bd31a70057ece32c3431abbc4ca1\\\', \\\'postprocesses/dac.py\\\': \\\'83ccbdaf5bab869921b5fa2f1c8a4f7dbadc96a8604e2ce9bb15406582e498de\\\', \\\'postprocesses/depth_consistency.py\\\': \\\'5768a8834c6fa98413eaed0b3274b6d179ed1e38a9974ceda008ef9ce2a9758a\\\', \\\'postprocesses/doppelgangers.py\\\': \\\'d875ab9e911d25f931df844f4171fb5f454cb61b63ffb0b16020404ec23694e0\\\', \\\'postprocesses/fast.py\\\': \\\'2a545fc28556a66d51782eab49086797aa3da0c414b62b407bd1543c9acd7762\\\', \\\'postprocesses/feature_track.py\\\': \\\'0e358eb2f9e515044aa515c2e132239e5b69f34ccd33e5fc9218943db426dc2a\\\', \\\'postprocesses/fsnet.py\\\': \\\'f03fdbe0a8b5ee11efee8c3032abb2f071a68767c452cbb8535dfe353afe86d0\\\', \\\'postprocesses/localize_pixloc.py\\\': \\\'f0aac5fe7ec3ee6a225188d5a70ca83289af3361b9e65479813464795f8d1521\\\', \\\'postprocesses/localize_sfm.py\\\': \\\'b340fb5a2f70943462b88b1c5ea99885d401f4329a0616936098ebd9131822d4\\\', \\\'postprocesses/matching_filter.py\\\': \\\'40b848720c39b5d538600f0101476f72e374a1b04114e97d52b534208df422c9\\\', \\\'postprocesses/minima_verifier.py\\\': \\\'f05a4a0273764be1dd7dc4e017e1828d6f599ecc250cce587e5a9032207884ab\\\', \\\'postprocesses/ncmnet.py\\\': \\\'3ad5e7cd7d1369f4b19f152de4eb5d2fd9fdcf9b7265195e02e94a879733dfad\\\', \\\'postprocesses/nms.py\\\': \\\'50be84db0d0c398ffd84a149be6908d27fc17616be7832aabc52737d520ecdbb\\\', \\\'postprocesses/panet.py\\\': \\\'0f168d16c060a1037f36b6b7246b2e1841d3c5f63e1979a0c862ef4d88050d4d\\\', \\\'postprocesses/pruning.py\\\': \\\'563db42f01b8fbf4a2828dec2586d475a20eb1dc07e82ac9cdb5707d668c091a\\\', \\\'postprocesses/ssc.py\\\': \\\'5873b1d375d683be45b011535d395441c10e6d223d0be33195c566511068e191\\\', \\\'postprocesses/vggt_verifier.py\\\': \\\'2bba280726b2b46c839369c0886168574ab3d1d36b0b023169550d27037f37cb\\\', \\\'preprocess.py\\\': \\\'388f0dc06e19834e4a9e1a90ae653a8950b2e56da7a29ff27c3a4596266bb4d5\\\', \\\'preprocesses/__init__.py\\\': \\\'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855\\\', \\\'preprocesses/config.py\\\': \\\'b9fd3d89ca08f4e20deaa5e671254f6d6a9083f2c281fbdbc625e777930c0812\\\', \\\'preprocesses/deblur.py\\\': \\\'733c84b0e9158e1c9111e6ebc9b865a8bfad49dc87f59897915b4b1542839c00\\\', \\\'preprocesses/depth.py\\\': \\\'b62f8e1e1724dd7e9ffc0e33de82b0810be2caee90d0e7ee0feb76393e4c2dda\\\', \\\'preprocesses/homography_adaptation.py\\\': \\\'c14aea1de38d48b152e4dc9f6b611610605180f218c97cea962ccee2bacf563b\\\', \\\'preprocesses/orientation.py\\\': \\\'32cbedbf5608b2fad4ee35c4f86086b6bff72b2b3522248b5a9fd339b4334840\\\', \\\'preprocesses/region.py\\\': \\\'a0fd0a4306e681b38d82eab2f543d58c92f8cd370a65d777a6ce1acf833fdac5\\\', \\\'preprocesses/segmentation.py\\\': \\\'9cbaef2ff382397ea29bb3a9ed826cd62bbc4fa25376cbf9511a5d3429cdc56b\\\', \\\'preprocesses/tile.py\\\': \\\'4b75bda1eedd9860cfa089315a36db5b739690527867de4449b160cd220c2f00\\\', \\\'preprocesses/visualizer.py\\\': \\\'b1566f1f4dafa0dc1895e1706f7ce662859f81d5751c89702371a7746b31c137\\\', \\\'retrievers/base.py\\\': \\\'cdbb2f017335bcf875482f51774e8348eeccb79f286a81ba8c343fd2effa4016\\\', \\\'retrievers/config.py\\\': \\\'da8e49eaf46ebd2aa0ab6ef892f5575e30d93bba50b0ce4067fb0a41833cb095\\\', \\\'retrievers/factory.py\\\': \\\'48f13ca3a459f45c5a3e2458000243873e2788abe31121ca0bd8c4a5f15d18ea\\\', \\\'retrievers/mast3r_retrieval_asmk.py\\\': \\\'e130be593359a352c2c68e662a4e6889a8d8e0c3d3c6a4c486b949801e8b0745\\\', \\\'shortlists/__init__.py\\\': \\\'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855\\\', \\\'shortlists/ann.py\\\': \\\'d8d1646411b36e836279a52bbb82146283a34c19cedf29a6e4c600471aedf134\\\', \\\'shortlists/base.py\\\': \\\'a4aae65ceb6ad5fa99e84a868f855864aaf28c9586d097f046115dd34110d319\\\', \\\'shortlists/config.py\\\': \\\'d5ecfbdddf9c2a1bf9fe2753f6d10cc333e0acc2bae74bc59e798d396a148f5c\\\', \\\'shortlists/connected_component.py\\\': \\\'61f086d54c833d0fca6b8a50fc338aaaf744ea3b6dc67cb34bdc19190dbfb9fa\\\', \\\'shortlists/cvnet_rerank.py\\\': \\\'84117f70a6e83248769ac23be98427dc8d9b79ebb08bd42b313085921c737fab\\\', \\\'shortlists/doppelgangers.py\\\': \\\'26b903e3d3853c513b011c8ffe8665a074338468951871f855c6ec19cbc11d23\\\', \\\'shortlists/ensemble.py\\\': \\\'2588da5f89f8e0f0385cde2a15eb73f501bff1ada59ded5b27a6fa0328448e07\\\', \\\'shortlists/exhaustive.py\\\': \\\'e900b6fb32f8c3bfae7c6a8349bc409ce6e969765560f3fe42a576eaa3efc4cb\\\', \\\'shortlists/factory.py\\\': \\\'e98a4da7a47022fc32b91bfdf6892ff3f73205078395066e67f04487cd593ee8\\\', \\\'shortlists/global_descriptor.py\\\': \\\'dc8cb7de1bfa9c7ae6ff7892452c0356b5a72e9786f78966dd6efe9778fc3249\\\', \\\'shortlists/global_descriptor_fps.py\\\': \\\'79a6200338b72ebb45e4d6a186c54579bb7f441a3a02348415e8c5b3a7e5d154\\\', \\\'shortlists/imc2024.py\\\': \\\'fe72a8654fd33a28b22a7431185a825cbf59b2c3abc045a7655ffb8cc279b154\\\', \\\'shortlists/line2d_feature.py\\\': \\\'72178ca579c06822bd6ec2dea4c66832936ca32b6db5eb1d457f3a4021d9b6e0\\\', \\\'shortlists/mast3r_retrieval_asmk.py\\\': \\\'99c44d59b6de26c1a439bed1dbdcda6362941a828f793a9dae41440ec7775765\\\', \\\'shortlists/no_update.py\\\': \\\'c9436530644598198d7304763046a8d104b52e4b82dd0b816c1bb2104b730b3f\\\', \\\'shortlists/prematching.py\\\': \\\'bfe0491aa0d6923a2eb246c0dd21c9d28a976d0b9a77b56896d4a5a7261eb57f\\\', \\\'shortlists/prematching_topk.py\\\': \\\'c0787b8c5bcce053c3d865e450a30a2b39a4df45a8f6c461bda056ccab8f4b1d\\\', \\\'shortlists/random_walk.py\\\': \\\'b279f42b6e419351cdf6f9d171dffbc5fdf13db6696249cf65bb3c9b53cd63b1\\\', \\\'shortlists/visualizer.py\\\': \\\'e336f6bfdd37899e2ba01b6fee7a63e3acb610f777d0461532c28075f7a3ba39\\\', \\\'storage.py\\\': \\\'e784c866af46b76875ef48ae77b6503243b55bb574ceb7c33c87dc11e1c7f8b2\\\', \\\'utils/camvis.py\\\': \\\'dfdfc4fd30ce6cdaedef416ab3c237155476f6f022ded97740a8fb337a98238a\\\', \\\'utils/configuration.py\\\': \\\'37bc4fe3a3dbe2ad532a7f965645beaedf548a7f22fa0405e88bb34285ca6ab8\\\', \\\'utils/debug.py\\\': \\\'10d7950cbc30aa7d90c9bae6c1b198e1d928e05153f251712eaac076b36b8a73\\\', \\\'utils/hloc_io.py\\\': \\\'8e63e9fde4ad2e667237c53135835813075db0518bedde9daef91fefa556f0ca\\\', \\\'utils/imc25/database.py\\\': \\\'ce9907a12287c90b969f450c771b19622ea0d4331f15dad3af1e646eb45c4c4d\\\', \\\'utils/imc25/h5_to_db.py\\\': \\\'fb172fe4abf20fde06cab014fab1795c47a0feabd46848af4d330557f48a2ae9\\\', \\\'utils/imc25/metric.py\\\': \\\'ad2ce51407cc73ad82fed764a879374f27a932df55f7ac47c923936c00b68e8a\\\', \\\'utils/imc25/metric_before_bugfix.py\\\': \\\'f9bc19e74796b4083c52ad2606fccd48c19bec7b4bc1908e3e82a3696aceb501\\\', \\\'utils/metric.py\\\': \\\'548190c5c7cc246732fb8d358c701437e2c07eadd9ad2974b1d3e75f490283aa\\\', \\\'utils/vlad.py\\\': \\\'5b68308f55634c947edb5480305299072525d1766c4a3e4186caaa069d34b65c\\\', \\\'workspace.py\\\': \\\'5671785f1079671ff62a884b79b5f021ef70402706e0f2cb127d4831265ddfc0\\\'}\\nASSETS = {\\\'superpoint\\\': {\\\'filename\\\': \\\'superpoint_v1.pth\\\', \\\'sha256\\\': \\\'52b6708629640ca883673b5d5c097c4ddad37d8048b33f09c8ca0d69db12c40e\\\'}, \\\'aliked\\\': {\\\'filename\\\': \\\'aliked-n16.pth\\\', \\\'sha256\\\': \\\'5be8704840ed662d9d8c561bf7279c222092674e7eb05fd0feab94899e9d82f2\\\'}, \\\'mast3r\\\': {\\\'filename\\\': \\\'MASt3R_ViTLarge_BaseDecoder_512_catmlpdpt_metric.pth\\\', \\\'sha256\\\': \\\'e28f91b488554653e2b46ddae9c78c1143e0bcb2e27d3e26cdb0b717f1568eb2\\\'}, \\\'retrieval\\\': {\\\'filename\\\': \\\'MASt3R_ViTLarge_BaseDecoder_512_catmlpdpt_metric_retrieval_trainingfree.pth\\\', \\\'sha256\\\': \\\'000a324f06f9369ab3e32d03dd4b8a838d48e98e3f3bf9d7dd034dce1bfd4c50\\\'}, \\\'codebook\\\': {\\\'filename\\\': \\\'MASt3R_ViTLarge_BaseDecoder_512_catmlpdpt_metric_retrieval_codebook.pkl\\\', \\\'sha256\\\': \\\'700af9e31ad0ca6159955dc80b4fd6e14234780b52f5c91884f6553991933c1a\\\'}, \\\'isc\\\': {\\\'filename\\\': \\\'isc_ft_v107.pth.tar\\\', \\\'sha256\\\': \\\'512390a15ab52f8ab5adc14a315b157a5d6e732e3361c7a0dce5be6e95a86d30\\\'}, \\\'dinov2\\\': {\\\'filename\\\': \\\'pytorch_model.bin\\\', \\\'sha256\\\': \\\'014965d9e330e7f4bff8ddcbee9df5e4f2ca032b2f5180942a6edb454783e75d\\\', \\\'sidecars\\\': {\\\'config.json\\\': \\\'f7ff4cfa73d2f70647dbf6950541ad25d73082d54c2e7e9bded160c7656b2a70\\\', \\\'preprocessor_config.json\\\': \\\'14e780d86fa1861f8751f868d7f45425b5feb55c38ca26f152ca5097ab30f828\\\'}}}\\n\\nimport hashlib\\nimport json\\nimport os\\nimport platform\\nimport subprocess\\nimport sys\\nfrom pathlib import Path\\n\\nbase = Path("/content/ns64_clean")\\ndef digest(path):\\n    with path.open("rb") as stream:\\n        return hashlib.file_digest(stream,"sha256").hexdigest()\\n\\nrecord = {\\n    "status":"BLOCKED", "mechanism":"separate clean VM; no process sandbox",\\n    "scope":"pre-staging dependency and absence checks",\\n    "python":sys.version, "uid":os.getuid(),\\n    "limitations":["Physical VM deletion is operator-attested.",\\n                   "Absence checks do not prove read denial within this VM.",\\n                   "No opaque image package has been staged or verified yet."],\\n}\\ntry:\\n    reads = {}\\n    forbidden = [\\n        "/content/drive", "/content/ns64", "/kaggle/input", "/private.json",\\n        "/content/private.json", "/content/imc_ets_images.zip",\\n        "/content/imc_ns64_colab_migration.zip",\\n        "/content/ns64/project/configs/baselines/organizer_v1/development_scoring/ETs/train_labels.csv",\\n        "/content/ns64/ets_attempt/inference/predictions.csv",\\n    ]\\n    for name in forbidden:\\n        try:\\n            fd=os.open(name,os.O_RDONLY)\\n        except OSError as error:\\n            reads[name]={"accessible":False,"errno":error.errno}\\n        else:\\n            os.close(fd)\\n            reads[name]={"accessible":True}\\n    record["administrative_paths"]=reads\\n    kaggle = Path("/kaggle/input")\\n    if reads[str(kaggle)]["accessible"]:\\n        mounts = [line.split()[4] for line in Path("/proc/self/mountinfo").read_text().splitlines()]\\n        reads[str(kaggle)]["mount_at_path"] = str(kaggle) in mounts\\n        placeholder = (kaggle.resolve(strict=True) == kaggle and kaggle.is_dir()\\n                       and not any(kaggle.iterdir())\\n                       and not any(p == "/kaggle" or p.startswith("/kaggle/input/") for p in mounts))\\n        reads[str(kaggle)]["empty_input_directory"] = placeholder\\n    if any(v["accessible"] and not v.get("empty_input_directory", False) for v in reads.values()):\\n        raise ValueError("Administrative path accessible")\\n    source=base/"source"\\n    observed={p.relative_to(source).as_posix():digest(p) for p in source.rglob("*") if p.is_file() and p.suffix!=".pyc"}\\n    if any(p.is_symlink() for p in source.rglob("*")) or observed!=SOURCE_HASHES:\\n        raise ValueError("Published source inventory mismatch")\\n    record["source_inventory_hash"]=hashlib.sha256(json.dumps(observed,sort_keys=True).encode()).hexdigest()\\n    record["assets"]={}\\n    for key,asset in ASSETS.items():\\n        parent=base/"weights"/"dinov2" if key=="dinov2" else base/"weights"\\n        path=parent/asset["filename"]\\n        actual=digest(path)\\n        if actual!=asset["sha256"]:\\n            raise ValueError("Checkpoint mismatch: "+key)\\n        for name,expected in asset.get("sidecars",{}).items():\\n            if digest(parent/name)!=expected:\\n                raise ValueError("Processor configuration mismatch")\\n        record["assets"][key]=actual\\n    suspect=[]\\n    for directory in (Path("/content"),Path("/root")):\\n        for parent,dirs,files in os.walk(directory,followlinks=False):\\n            # Scope includes files outside the runtime; package caches can be large.\\n            dirs[:]=[n for n in dirs if n not in (".cache","node_modules")]\\n            for name in files:\\n                low=name.lower()\\n                if (low in ("train_labels.csv","pilot.json","private.json","predictions.csv")\\n                    or low.startswith("imc_baseline") or low.startswith("imc_ns64_ets_pilot")):\\n                    suspect.append(str(Path(parent)/name))\\n    record["unexpected_artifact_count"]=len(suspect)\\n    if suspect:\\n        raise ValueError("Unexpected administrative artifacts in VM")\\n    import torch,asmk.hamming,curope,pycolmap,kornia,mast3r.model,dust3r.inference\\n    if sys.version_info[:2]!=(3,11) or torch.__version__!="2.5.1+cu121" or not torch.cuda.is_available():\\n        raise RuntimeError("Runtime incompatible")\\n    record.update(device=torch.cuda.get_device_name(0),torch=torch.__version__,pycolmap=pycolmap.__version__,\\n                  dependencies="PASS",status="CLEAN_VM_PRECHECK_PASS")\\n    record["environment_freeze"]=subprocess.check_output([sys.executable,"-m","pip","freeze"],text=True)\\n    record["gpu"]=subprocess.check_output(["nvidia-smi","--query-gpu=name,driver_version,memory.total","--format=csv,noheader"],text=True).strip()\\n    mountinfo=Path("/proc/self/mountinfo").read_text().splitlines()\\n    record["mountpoints"]=[line.split()[4] for line in mountinfo]\\nexcept Exception as error:\\n    record.update(error_type=type(error).__name__,error=str(error))\\nfinally:\\n    record["probe_sha256"]=digest(Path(__file__))\\n    output=Path.cwd()/"environment.json"\\n    output.write_text(json.dumps(record,indent=2))\\n    print(json.dumps({k:v for k,v in record.items() if k not in ("environment_freeze","mountpoints","assets","administrative_paths")},indent=2))\\n    print("Saved",output)\\n\',{\'__file__\':__file__})\nenvironment=json.loads((work/\'environment.json\').read_text())\nrequire(environment[\'status\']==\'CLEAN_VM_PRECHECK_PASS\',\'Access or integrity check failed\')\nrequire(hashlib.sha256(environment[\'environment_freeze\'].encode()).hexdigest()==\'8bb71679bf5735f6e3c8225885be88c1f530639094df310a5a29e6691498455b\' and environment[\'gpu\']==\'Tesla T4, 580.82.07, 15360 MiB\',\'Dependency or GPU drift since approved run\')\nrequire(seed==20261111,\'Unauthorized seed\')\nfor path in (\'/content/ns64_imc2023_haiper_run\',\'/content/imc2023_haiper_opaque.zip\',\'/content/ns64_imc2023_theather_imc2024_church_run\',\'/content/imc2023_theather_imc2024_church_opaque.zip\',\'/content/imc2023_heritage\',\'/content/ns64_stairs_run\',\'/content/stairs_opaque.zip\',\'/content/ns64_neutral_runs\',\'/content/opaque_inputs.zip\',\'/content/ns64/project\',\'/proc/1/root/content/drive\',\'/var/run/docker.sock\',\'/content/imc2024_dioscuri_baalshamin\',\'/content/imc_ets_images.zip\'):\n    require(not os.path.lexists(path),\'Unexpected administrative or prior run path \'+path)\nsave(\'boundary.json\',dict(status=\'PASS\', path_checks=environment[\'administrative_paths\'], separate_clean_runtime=True))\nimport pandas as pd, torch, pycolmap, yaml\nrequire(sys.version_info[:3]==(3,11,11) and torch.version.cuda==\'12.1\' and pycolmap.__version__==\'3.11.1\' and torch.cuda.get_device_name(0)==\'Tesla T4\',\'Runtime drift\')\nassets=base/\'assets.json\'\nrequire(digest(assets)==ASSET_MANIFEST,\'Asset alias manifest mismatch\')\ndefinitions=yaml.safe_load((source/\'conf/models.yaml\').read_text())\nfor key,asset in json.loads(assets.read_text()).items():\n    path=base/\'weights\'/(\'dinov2\' if key==\'dinov2\' else asset[\'filename\'])\n    definitions[asset[\'alias\']]={\'kernel\':str(path),\'local\':str(path)}\nsave(\'models.json\',definitions)\nos.environ.update(DEFAULT_MODEL_LIST_PATH=str(work/\'models.json\'),DEFAULT_EXTRA_DIR=str(work/\'extra\'),HF_HUB_OFFLINE=\'1\',TRANSFORMERS_OFFLINE=\'1\',MPLBACKEND=\'Agg\')\nnames=[f\'image_{i:04d}.png\' for i in range(138)]\nimages=work/\'test/imc2024_dioscuri_baalshamin\'\nimages.mkdir(parents=True)\nwith zipfile.ZipFile(Path(\'/content/imc2024_dioscuri_baalshamin_opaque.zip\')) as archive:\n    save(\'input_receipt.json\',dict(archive_sha256=digest(Path(\'/content/imc2024_dioscuri_baalshamin_opaque.zip\')),permutation_sha256=PINS[seed]))\n    require(sorted(archive.namelist())==names,\'Wrong archive inventory\')\n    for name in names:\n        (images/name).write_bytes(archive.read(name))\nrequire(hashlib.sha256(chr(10).join(digest(images/n) for n in names).encode()).hexdigest()==PINS[seed],\'Extracted order mismatch\')\nsys.path.insert(0,str(source))\nfrom config import PipelineConfig\nfrom pipeline import create_pipeline\nfrom data import IMC2025TestData\nimport pipelines.imc2025_pipeline as impl\nconfig=source/\'conf/pipeline/imc2025/mast3rhybrid/mast3rhybrid-022-c.yaml\'\nsave(\'identity.json\',dict(benchmark={\'benchmark_hash\': \'a78f0542413b8ac2ec8c262988ed5303c934490a2b31d2a45d3939dd6d3ab2a4\', \'split_hash\': \'170546743202f21b9bcc98376ff09a6eae2da16ce833115478a976c4f46b2057\', \'evaluator_hash\': \'c7ae94306ba0ae4ce9c00e9abc7e15bd99f3091fa063b5826b0dd567a6c3f261\', \'threshold_hash\': \'c6aec2fc2c8f9cb5f268f57333911f0835fcaba8aedc0eec67951ed729e68645\'},method_id=\'ns64_mast3rhybrid_022c\',comparison_baseline_id=\'organizer_baseline_v1_dev_full\',experiment_id=f\'ns64_isolated_v1_imc2024_dioscuri_baalshamin_{seed}\',seed=seed,permutation_sha256=PINS[seed],config_sha256=digest(config),source_revision=\'07b5695b99a8ae4725d5d952f0065c2157fadd7d\'))\nframe=pd.DataFrame([dict(image_id=str(i),dataset=\'imc2024_dioscuri_baalshamin\',scene=\'cluster0\',image=n,rotation_matrix=\'\',translation_vector=\'\') for i,n in enumerate(names)])\ndef peaks():\n    return dict(worker_peak_rss_kib=resource.getrusage(resource.RUSAGE_SELF).ru_maxrss,peak_gpu_allocated_bytes=torch.cuda.max_memory_allocated(),peak_gpu_reserved_bytes=torch.cuda.max_memory_reserved())\ndef measure(fn,name):\n    @functools.wraps(fn)\n    def wrapper(*args,**kwargs):\n        start=time.monotonic()\n        row=dict(stage=name,status=\'FAILED\')\n        reserve={\'shortlist_generator\':6600,\'run_point_tracking_matching_stage\':5520,\'verify_matches\':5520}.get(name,1200)\n        remaining=float(os.environ[\'STAIRS_DEADLINE\'])-start\n        save(\'current_stage.json\',dict(stage=name,remaining_seconds=remaining,reserve_seconds=reserve,**peaks()))\n        require(remaining>reserve,\'Stage start deadline exceeded: \'+name)\n        try:\n            if name in (\'shortlist_generator\',\'run_point_tracking_matching_stage\',\'incremental_mapping\'):\n                import shutil\n                free_gpu,total_gpu=torch.cuda.mem_get_info()\n                available=int(next(l.split()[1] for l in Path(\'/proc/meminfo\').read_text().splitlines() if l.startswith(\'MemAvailable:\')))\n                row.update(free_gpu_bytes=free_gpu,available_ram_kib=available,disk_free=shutil.disk_usage(work).free,remaining_seconds=remaining)\n                require(available>1048576 and row[\'disk_free\']>1073741824,\'Resource headroom exhausted\')\n            if name==\'incremental_mapping\':\n                remaining=float(os.environ[\'STAIRS_DEADLINE\'])-time.monotonic()\n                row[\'remaining_before_sfm\']=remaining\n                require(remaining>5520,\'Insufficient time for pair-scaled SfM planning and 20-minute export reserve\')\n            result=fn(*args,**kwargs)\n            if name in (\'shortlist_generator\',\'incremental_mapping\'):\n                row[\'count\']=len(result)\n            if name==\'fuse_matching_sets_late\':\n                row[\'retained_pairs\']=sum(len(group) for group in result[1].matches.values())\n            if name==\'verify_matches\':\n                row[\'verified_pairs\']=result.count\n                save(\'verified_edges.json\',[[str(i),str(j)] for i,g in result.inliers.items() for j,v in g.items() if len(v)])\n            row[\'status\']=\'PASS\'\n            return result\n        finally:\n            row.update(seconds=time.monotonic()-start,**peaks())\n            checkpoint=work/f\'checkpoint_{name}.json\'\n            with checkpoint.open(\'x\') as stream:\n                json.dump(row,stream)\n            with (work/\'stages.jsonl\').open(\'a\') as log:\n                print(json.dumps(row),file=log)\n    return wrapper\ntry:\n    import shutil\n    free_vram,total_vram=torch.cuda.mem_get_info()\n    ram_kib=int(next(x.split()[1] for x in Path(\'/proc/meminfo\').read_text().splitlines() if x.startswith(\'MemAvailable:\')))\n    disk_free=shutil.disk_usage(work).free\n    scratch_estimate=max(1024**3,4*sum(p.stat().st_size for p in images.iterdir()))\n    save(\'preflight.json\',dict(gpu=torch.cuda.get_device_name(),free_vram=free_vram,total_vram=total_vram,available_ram_kib=ram_kib,disk_free=disk_free,planning_scratch_bytes=scratch_estimate,candidate_pairs=9453,remaining_seconds=float(os.environ[\'STAIRS_DEADLINE\'])-time.monotonic()))\n    require(free_vram>=9412018176 and ram_kib>=8388608 and disk_free>=scratch_estimate,\'Below pilot-based resource planning floor\')\n    import gc,pickle\n    loaded=[]\n    for key,asset in json.loads(assets.read_text()).items():\n        folder=base/\'weights\'/(\'dinov2\' if key==\'dinov2\' else \'\')\n        path=folder/asset[\'filename\']\n        if path.suffix==\'.pkl\':\n            with path.open(\'rb\') as stream:\n                payload=pickle.load(stream)\n        else:\n            payload=torch.load(path,map_location=\'cpu\',weights_only=False)\n        require(payload is not None,\'Empty checkpoint \'+key)\n        del payload\n        gc.collect()\n        loaded.append(key)\n        save(\'weight_load.json\',dict(loaded=loaded))\n    pipeline=create_pipeline(PipelineConfig.load_config(config),device=torch.device(\'cuda:0\'))\n    with contextlib.ExitStack() as stack:\n        for owner,name in [(pipeline,\'shortlist_generator\'),(impl,\'run_local_feature_extraction_and_matching_stage\'),(impl,\'run_detector_free_matching_stage\'),(impl,\'run_point_tracking_matching_stage\'),(impl,\'fuse_matching_sets_late\'),(impl,\'verify_matches\'),(impl.pycolmap,\'incremental_mapping\')]:\n            stack.enter_context(patch.object(owner,name,measure(getattr(owner,name),name)))\n        result=pipeline.run(frame,IMC2025TestData(frame,data_root_dir=work))\n    require(sorted(result.image)==names and set(result.dataset)=={\'imc2024_dioscuri_baalshamin\'},\'Invalid prediction coverage\')\n    result[[\'dataset\',\'scene\',\'image\',\'rotation_matrix\',\'translation_vector\']].to_csv(work/\'opaque.csv\',index=False)\n    outliers=result.scene==\'outliers\'\n    save(\'prediction.json\',dict(sha256=digest(work/\'opaque.csv\'),registered=int((~outliers).sum()),clusters=int(result.loc[~outliers,\'scene\'].nunique()),outliers=int(outliers.sum())))\nfinally:\n    save(\'resources.json\',dict(**peaks(),work_bytes=sum(p.stat().st_size for p in work.rglob(\'*\') if p.is_file()),mem_available=next(line for line in Path(\'/proc/meminfo\').read_text().splitlines() if line.startswith(\'MemAvailable:\'))))\n'
from pathlib import Path
from google.colab import files
import os, sys, signal, subprocess, time, json, zipfile, hashlib
content=Path('/content')
python=content/'ns64_clean/venv/bin/python'
assert python.is_file(), 'Use the verified Phase 3C.4 session.'
batch=content/'ns64_imc2024_dioscuri_baalshamin_run'
batch.mkdir(exist_ok=True)
def run_seed(seed):
    budget=min(14400,int(input('Available session seconds now (maximum 14400): ')))
    if budget<8400: raise ValueError('Insufficient session time; no attempt started')
    work=batch/str(seed)
    archive=content/f'imc_ns64_imc2024_dioscuri_baalshamin_worker_{seed}.zip'
    if work.exists() or archive.exists():
        raise RuntimeError(f'Seed {seed} already has an attempt. Preserve its files.')
    work.mkdir()
    script=work/'worker.py'
    script.write_text(WORKER)
    assert hashlib.sha256(script.read_bytes()).hexdigest()=='9bf25783cb174eec0d702f1d18a942284b7853a735601f498cba23ecbff7c73d', 'Worker integrity mismatch'
    start=time.monotonic()
    status='FAILED'
    try:
        with (work/'worker.log').open('w') as log:
            process=subprocess.Popen([str(python),'-I','-u',str(script),str(seed)],cwd=work,stdout=log,stderr=subprocess.STDOUT,start_new_session=True,env=dict(os.environ,STAIRS_DEADLINE=str(start+budget)))
            try:
                while process.poll() is None:
                    left=start+budget-time.monotonic()
                    stage_path=work/'current_stage.json'
                    reserve=json.loads(stage_path.read_text())['reserve_seconds'] if stage_path.exists() else 1200
                    if left<=reserve:
                        raise subprocess.TimeoutExpired('stage deadline',budget-reserve)
                    time.sleep(min(1,left-reserve))
                code=process.returncode
                status='PASS' if code==0 else 'FAILED'
            finally:
                if process.poll() is None:
                    os.killpg(process.pid,signal.SIGKILL)
                    process.wait()
    except subprocess.TimeoutExpired:
        status='TIMEOUT'
    finally:
        result=dict(status=status,seed=seed,experiment_id=f'ns64_isolated_v1_imc2024_dioscuri_baalshamin_{seed}',seconds=time.monotonic()-start,budget_seconds=budget,worker_sha256=hashlib.sha256(script.read_bytes()).hexdigest())
        (work/'run.json').write_text(json.dumps(result,indent=2))
        with zipfile.ZipFile(archive,'x',zipfile.ZIP_DEFLATED) as z:
            for checkpoint in sorted(work.glob('checkpoint_*.json')):
                z.write(checkpoint,checkpoint.name)
            for name in ('input_receipt.json','weight_load.json','preflight.json','current_stage.json','verified_edges.json','run.json','environment.json','boundary.json','worker.log','stages.jsonl','identity.json','resources.json','prediction.json','opaque.csv'):
                p=work/name
                if p.is_file():
                    z.write(p,name)
        print(result,flush=True)
        files.download(str(archive))
    return status

assert run_seed(20261111)=='PASS'